# Machine Learning para MS-CFLP-CI (Multi-Site Capacitated Facility Location Problem with Incompatibility Constraints)

### Descripción del problema

Dado un conjunto de almacenes candidatos y tiendas, hay que decidir:
1. Qué almacenes abrir (pagando un costo fijo por cada uno).
2. A qué almacén asignar cada tienda (pagando costo de suministro × demanda).

Restricciones:
- La suma de demandas asignadas a cada almacén no puede superar su capacidad.
- Algunos pares de tiendas son **incompatibles**: no pueden asignarse al mismo almacén.

El objetivo es minimizar el costo total (costos fijos + costos de suministro).

## Imports

Librerías estándar, `numpy` y `sklearn` (regresión, árboles, random forest).

In [333]:
import io
import contextlib
import random
import re
import sys
import math
import time
from dataclasses import dataclass
from typing import TypedDict

import numpy as np
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.tree import DecisionTreeRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel, WhiteKernel


### Estructuras de datos

`DatosCFLP` es un `TypedDict` con los campos del problema leídos desde el `.dzn`.
Las soluciones se representan como `(abierta: list[bool], asignada: list[int], restante: list[int])`.
La lista `restante` actúa como caché de capacidades residuales: cada comprobación de factibilidad
es O(1) en lugar de O(n).

In [334]:
@dataclass
class ResultadoBenchmark:
    nombre: str
    costo_final: float
    tiempo_total_s: float
    tiempo_entrenamiento_ml_s: float = 0.0
    tiempo_inferencia_ml_s: float = 0.0
    modelo_ml: object = None

    @property
    def tiempo_algoritmo_puro_s(self) -> float:
        return (
            self.tiempo_total_s
            - self.tiempo_entrenamiento_ml_s
            - self.tiempo_inferencia_ml_s
        )

    @property
    def overhead_ml_pct(self) -> float:
        overhead = self.tiempo_entrenamiento_ml_s + self.tiempo_inferencia_ml_s
        return overhead / self.tiempo_total_s * 100 if self.tiempo_total_s else 0.0

In [335]:
class DatosCFLP(TypedDict):
    numero_instalaciones: int
    numero_clientes: int
    capacidad_instalacion: list[int]
    costo_fijo_instalacion: list[int]
    demanda_cliente: list[int]
    costo_envio: list[list[int]]
    pares_incompatibles: list[tuple[int, int]]


SolucionCFLP = tuple[list[bool], list[int], list[int]]

## Lectura de instancia (`.dzn`)

In [336]:
def leer_instancia_dzn(ruta_archivo: str) -> DatosCFLP:
    """Lee una instancia MS-CFLP-CI desde un fichero MiniZinc (.dzn)."""
    with open(ruta_archivo, "r") as archivo:
        contenido_completo = archivo.read()

    numero_instalaciones = int(
        extraer_valor_escalar(contenido_completo, "Warehouses")
    )

    numero_clientes = int(
        extraer_valor_escalar(contenido_completo, "Stores")
    )

    capacidad_instalacion = extraer_lista_enteros(
        contenido_completo, "Capacity"
    )

    costo_fijo_instalacion = extraer_lista_enteros(
        contenido_completo, "FixedCost"
    )

    demanda_cliente = extraer_lista_enteros(contenido_completo, "Goods")

    # SupplyCost[tienda][almacén] → transponer a [almacén][tienda]
    costo_envio_raw = extraer_matriz_enteros(
        contenido_completo, "SupplyCost",
        numero_clientes, numero_instalaciones
    )
    costo_envio = [list(col) for col in zip(*costo_envio_raw)]

    pares_incompatibles = extraer_pares_incompatibles(contenido_completo)

    return {
        "numero_instalaciones": numero_instalaciones,
        "numero_clientes": numero_clientes,
        "capacidad_instalacion": capacidad_instalacion,
        "costo_fijo_instalacion": costo_fijo_instalacion,
        "demanda_cliente": demanda_cliente,
        "costo_envio": costo_envio,
        "pares_incompatibles": pares_incompatibles,
    }



def extraer_valor_escalar(texto: str, nombre_variable: str) -> str:
    """Extrae un escalar entero de `NombreVariable = 50;`."""
    patron = rf"{nombre_variable}\s*=\s*([0-9]+)\s*;"
    coincidencia = re.search(patron, texto)
    if not coincidencia:
        raise ValueError(
            f"No se encontró la variable '{nombre_variable}' en el archivo."
        )
    return coincidencia.group(1)



def extraer_lista_enteros(texto: str, nombre_variable: str) -> list[int]:
    """Extrae una lista de enteros de `NombreVariable = [v1, v2, ...];`."""
    patron = rf"{nombre_variable}\s*=\s*\[([^\]]+)\]"
    coincidencia = re.search(patron, texto, re.DOTALL)
    if not coincidencia:
        raise ValueError(
            f"No se encontró la lista '{nombre_variable}' en el archivo."
        )
    valores_texto = coincidencia.group(1)
    return [int(v.strip()) for v in valores_texto.split(",") if v.strip()]



def extraer_matriz_enteros(
    texto: str,
    nombre_variable: str,
    numero_filas: int,
    numero_columnas: int,
) -> list[list[int]]:
    """Extrae una matriz de enteros en formato MiniZinc `[| ... |]`."""
    patron = rf"{nombre_variable}\s*=\s*\[([^\]]+)\]"
    coincidencia = re.search(patron, texto, re.DOTALL)
    if not coincidencia:
        raise ValueError(
            f"No se encontró la matriz '{nombre_variable}' en el archivo."
        )

    bloque = coincidencia.group(1)
    filas_texto = bloque.split("|")

    matriz = []
    for fila_texto in filas_texto:
        fila_texto = fila_texto.strip().strip(",").strip()
        if not fila_texto:
            continue
        valores_fila = [
            int(v.strip()) for v in fila_texto.split(",") if v.strip()
        ]
        if valores_fila:
            matriz.append(valores_fila)

    return matriz



def extraer_pares_incompatibles(texto: str) -> list[tuple[int, int]]:
    """Extrae pares incompatibles de `IncompatiblePairs = [|a, b | ...|];`.
    Los índices en el archivo son 1-based; se convierten a 0-based.
    """
    patron = r"IncompatiblePairs\s*=\s*\[([^\]]+)\]"
    coincidencia = re.search(patron, texto, re.DOTALL)
    if not coincidencia:
        return []
    bloque = coincidencia.group(1)
    pares = []
    for fila in bloque.split("|"):
        fila = fila.strip().strip(",").strip()
        if not fila:
            continue
        partes = [v.strip() for v in fila.split(",") if v.strip()]
        if len(partes) >= 2:
            pares.append((int(partes[0]) - 1, int(partes[1]) - 1))
    return pares



def _precomputar_incompatibles(
    numero_clientes: int,
    pares_incompatibles: list[tuple[int, int]],
) -> list[set[int]]:
    """Convierte lista de pares en lista de conjuntos de incompatibles por cliente."""
    incompatibles: list[set[int]] = [set() for _ in range(numero_clientes)]
    for a, b in pares_incompatibles:
        incompatibles[a].add(b)
        incompatibles[b].add(a)
    return incompatibles

## Función de costo total

In [337]:
def calcular_costo_total(
    instalacion_esta_abierta: list[bool],
    instalacion_asignada_al_cliente: list[int],
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
) -> int:
    """Costo fijo de instalaciones abiertas + costo de envío por cliente."""
    costo_por_abrir_instalaciones = sum(
        costo_fijo_instalacion[i]
        for i, abierta in enumerate(instalacion_esta_abierta)
        if abierta
    )

    costo_por_envios_a_clientes = sum(
        demanda_cliente[j]
        * costo_envio[instalacion_asignada_al_cliente[j]][j]
        for j in range(len(instalacion_asignada_al_cliente))
    )

    return costo_por_abrir_instalaciones + costo_por_envios_a_clientes


## 1. Búsqueda local

Tres movimientos aparecen una y otra vez a lo largo del cuaderno: reasignar
un cliente (N1), cerrar una instalación (N2) y abrir una instalación (N3).
GRASP, GVNS y SA los usan todos, cada uno con su propia política de
aceptación (barrido completo, primera mejora, o Metropolis), pero el cálculo
del delta de coste de cada movimiento es exactamente el mismo cálculo en los
tres casos. Se define una sola vez aquí, y cada algoritmo posterior solo
aporta su política de cuándo aplicarlo.

Estas funciones **no aplican el movimiento ni deciden si conviene**: solo
calculan el delta de coste (y, en N2/N3, qué reasignación concreta lo
produce). Quien llama decide — GRASP/GVNS aplican solo si el delta es
negativo (rentable); SA le pasa el delta, sea cual sea su signo, al criterio
de Metropolis.

### N1: Reasignar un cliente

Reasignar un cliente de su instalación actual a otra factible. El delta de
coste tiene tres componentes: la diferencia de coste de envío entre origen y
destino, el coste fijo de abrir la instalación destino si estaba cerrada, y
el ahorro de coste fijo si el cliente era el único que quedaba en la
instalación de origen (con lo que esta cierra).

In [338]:
def _delta_reasignar_cliente(
    indice_cliente: int,
    instalacion_actual: int,
    instalacion_destino: int,
    instalacion_esta_abierta: list[bool],
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    es_el_unico_en_origen: bool,
) -> int:
    """N1: delta de coste de mover `indice_cliente` desde `instalacion_actual`
    hasta `instalacion_destino`. Asume que el destino ya se comprobó factible
    (capacidad e incompatibilidades) antes de llamar; esta función solo
    calcula el delta, no aplica el movimiento ni decide si conviene."""
    demanda = demanda_cliente[indice_cliente]
    costo_envio_actual = costo_envio[instalacion_actual][indice_cliente] * demanda
    costo_envio_nuevo  = costo_envio[instalacion_destino][indice_cliente] * demanda
    ahorro_cierre_origen = (
        costo_fijo_instalacion[instalacion_actual] if es_el_unico_en_origen else 0
    )
    coste_apertura_destino = (
        costo_fijo_instalacion[instalacion_destino]
        if not instalacion_esta_abierta[instalacion_destino] else 0
    )
    return costo_envio_nuevo + coste_apertura_destino - costo_envio_actual - ahorro_cierre_origen

def _reasignar_clientes_en_barrido(
    instalacion_esta_abierta: list[bool],
    instalacion_asignada_al_cliente: list[int],
    capacidad_restante_instalacion: list[int],
    numero_instalaciones: int,
    numero_clientes: int,
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    incompatibles_cliente: list[set[int]] | None = None,
) -> tuple[SolucionCFLP, bool]:
    """N1 aplicado: barre todos los clientes, reasignando cada uno a la primera
    alternativa que reduzca el coste (puede reasignar al mismo cliente más de
    una vez dentro del mismo barrido, si tras moverlo aparece otra alternativa
    mejor). Usada tanto por GRASP (búsqueda local completa) como por GVNS
    (como vecindario N1 del VND)."""
    instalacion_esta_abierta = list(instalacion_esta_abierta)
    instalacion_asignada_al_cliente = list(instalacion_asignada_al_cliente)
    capacidad_restante_instalacion = list(capacidad_restante_instalacion)

    hubo_mejora = False

    for indice_cliente in range(numero_clientes):
        instalacion_actual = instalacion_asignada_al_cliente[indice_cliente]
        demanda = demanda_cliente[indice_cliente]
        inc = incompatibles_cliente[indice_cliente] if incompatibles_cliente else set()

        es_el_unico = sum(
            1 for j in range(numero_clientes)
            if instalacion_asignada_al_cliente[j] == instalacion_actual
        ) == 1

        for instalacion_destino in range(numero_instalaciones):
            if instalacion_destino == instalacion_actual:
                continue
            if capacidad_restante_instalacion[instalacion_destino] < demanda:
                continue
            if any(instalacion_asignada_al_cliente[k] == instalacion_destino for k in inc):
                continue

            variacion = _delta_reasignar_cliente(
                indice_cliente, instalacion_actual, instalacion_destino,
                instalacion_esta_abierta, costo_fijo_instalacion, demanda_cliente,
                costo_envio, es_el_unico,
            )

            if variacion < 0:
                capacidad_restante_instalacion[instalacion_actual] += demanda
                if es_el_unico:
                    instalacion_esta_abierta[instalacion_actual] = False
                instalacion_asignada_al_cliente[indice_cliente] = instalacion_destino
                capacidad_restante_instalacion[instalacion_destino] -= demanda
                instalacion_esta_abierta[instalacion_destino] = True

                instalacion_actual = instalacion_destino
                es_el_unico = False
                hubo_mejora = True

    return (
        instalacion_esta_abierta,
        instalacion_asignada_al_cliente,
        capacidad_restante_instalacion,
    ), hubo_mejora


### N2 Cerrar una instalación

Cerrar una instalación abierta, reasignando a todos sus clientes en la
alternativa factible más barata disponible en ese momento. El delta acumula,
cliente a cliente, el incremento en coste de envío —más el coste fijo de
abrir una alternativa que estuviera cerrada, contado una sola vez aunque
varios clientes se reasignen a ella— menos el ahorro del coste fijo de la
instalación que se cierra.

In [339]:
def _evaluar_cierre_instalacion(
    instalacion: int,
    instalacion_esta_abierta: list[bool],
    instalacion_asignada_al_cliente: list[int],
    capacidad_restante_instalacion: list[int],
    numero_instalaciones: int,
    numero_clientes: int,
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    incompatibles_cliente: list[set[int]],
) -> tuple[int, dict[int, int]] | None:
    """N2: evalúa el delta de coste de cerrar `instalacion`, reasignando a sus
    clientes en la alternativa factible más barata. Devuelve (delta,
    reasignacion) o None si el cierre no es factible (algún cliente sin
    alternativa)."""
    clientes_asignados = [
        j for j in range(numero_clientes)
        if instalacion_asignada_al_cliente[j] == instalacion
    ]
    if not clientes_asignados:
        return None

    ahorro_al_cerrar = costo_fijo_instalacion[instalacion]
    incremento_en_envios = 0
    reasignacion: dict[int, int] = {}
    demanda_comprometida: dict[int, int] = {}
    # Instalaciones cerradas que ya reciben a algún cliente de esta misma
    # evaluación: su coste fijo de apertura se paga una sola vez, no una vez
    # por cliente que se reasigna a ellas.
    instalaciones_recien_abiertas: set[int] = set()

    for cliente in clientes_asignados:
        d = demanda_cliente[cliente]
        costo_actual = costo_envio[instalacion][cliente] * d
        inc = incompatibles_cliente[cliente]
        prohibidas = {instalacion_asignada_al_cliente[k] for k in inc}
        prohibidas |= {reasignacion[k] for k in inc if k in reasignacion}

        mejor_alt, mejor_costo_alt = None, float("inf")
        for i_alt in range(numero_instalaciones):
            if i_alt == instalacion or i_alt in prohibidas:
                continue
            cap_disponible = (
                capacidad_restante_instalacion[i_alt] - demanda_comprometida.get(i_alt, 0)
            )
            if cap_disponible < d:
                continue
            c = costo_envio[i_alt][cliente] * d
            if not instalacion_esta_abierta[i_alt] and i_alt not in instalaciones_recien_abiertas:
                c += costo_fijo_instalacion[i_alt]
            if c < mejor_costo_alt:
                mejor_costo_alt, mejor_alt = c, i_alt

        if mejor_alt is None:
            return None

        incremento_en_envios += mejor_costo_alt - costo_actual
        reasignacion[cliente] = mejor_alt
        demanda_comprometida[mejor_alt] = demanda_comprometida.get(mejor_alt, 0) + d
        if not instalacion_esta_abierta[mejor_alt]:
            instalaciones_recien_abiertas.add(mejor_alt)

    delta = incremento_en_envios - ahorro_al_cerrar
    return delta, reasignacion

def _cerrar_instalaciones_poco_rentables(
    instalacion_esta_abierta: list[bool],
    instalacion_asignada_al_cliente: list[int],
    capacidad_restante_instalacion: list[int],
    numero_instalaciones: int,
    numero_clientes: int,
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    incompatibles_cliente: list[set[int]] | None = None,
) -> tuple[SolucionCFLP, bool]:
    """N2 aplicado: cierra todas las instalaciones abiertas donde el ahorro
    fijo supera el incremento en envíos. Usada tanto por GRASP (búsqueda
    local completa) como por GVNS (como vecindario N2 del VND)."""
    instalacion_esta_abierta = list(instalacion_esta_abierta)
    instalacion_asignada_al_cliente = list(instalacion_asignada_al_cliente)
    capacidad_restante_instalacion = list(capacidad_restante_instalacion)
    incompatibles = incompatibles_cliente or [set() for _ in range(numero_clientes)]

    hubo_mejora = False

    for indice_instalacion_a_cerrar in range(numero_instalaciones):
        if not instalacion_esta_abierta[indice_instalacion_a_cerrar]:
            continue

        propuesta = _evaluar_cierre_instalacion(
            indice_instalacion_a_cerrar,
            instalacion_esta_abierta, instalacion_asignada_al_cliente,
            capacidad_restante_instalacion, numero_instalaciones, numero_clientes,
            costo_fijo_instalacion, demanda_cliente, costo_envio, incompatibles,
        )
        if propuesta is None:
            continue
        delta, reasignacion = propuesta

        if delta < 0:
            for ic, nueva_instalacion in reasignacion.items():
                demanda_ic = demanda_cliente[ic]
                capacidad_restante_instalacion[indice_instalacion_a_cerrar] += demanda_ic
                instalacion_asignada_al_cliente[ic] = nueva_instalacion
                capacidad_restante_instalacion[nueva_instalacion] -= demanda_ic
                instalacion_esta_abierta[nueva_instalacion] = True
            instalacion_esta_abierta[indice_instalacion_a_cerrar] = False
            hubo_mejora = True

    return (
        instalacion_esta_abierta,
        instalacion_asignada_al_cliente,
        capacidad_restante_instalacion,
    ), hubo_mejora


### N3: Abrir una instalación cerrada

Abrir una instalación cerrada, migrando a ella los clientes que más ahorran
enviándose desde allí en vez de su instalación actual, hasta agotar su
capacidad. El delta resta al coste fijo de abrir el ahorro de envío
acumulado, y también el coste fijo de cualquier instalación de origen que se
quede sin clientes y por tanto cierre como efecto colateral —sin este
término se subestimaría el beneficio real de abrir.

In [340]:
def _evaluar_apertura_instalacion(
    instalacion: int,
    instalacion_esta_abierta: list[bool],
    instalacion_asignada_al_cliente: list[int],
    capacidad_restante_instalacion: list[int],
    numero_clientes: int,
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    incompatibles_cliente: list[set[int]],
    carga_por_instalacion: list[int],
) -> tuple[int, list[int]] | None:
    """N3: evalúa el delta de coste de abrir `instalacion`, migrando a ella el
    mejor conjunto de clientes por ahorro de envío. Devuelve (delta,
    seleccionados) o None si ningún cliente se beneficia de moverse.

    `carga_por_instalacion` es el número de clientes que tiene asignados cada
    instalación en la solución actual: si todos los clientes de una
    instalación de origen resultan seleccionados, esa instalación se queda
    sin clientes y cierra, ahorrando su coste fijo.
    """
    candidatos_ahorro = []
    for cliente in range(numero_clientes):
        actual = instalacion_asignada_al_cliente[cliente]
        d = demanda_cliente[cliente]
        ahorro = costo_envio[actual][cliente] * d - costo_envio[instalacion][cliente] * d
        if ahorro > 0:
            candidatos_ahorro.append((ahorro, cliente))

    if not candidatos_ahorro:
        return None

    candidatos_ahorro.sort(reverse=True)

    seleccionados: list[int] = []
    capacidad_disponible = capacidad_restante_instalacion[instalacion]
    ahorro_acumulado = 0
    excluidos_acumulados: set[int] = set()

    for ahorro, cliente in candidatos_ahorro:
        d = demanda_cliente[cliente]
        if d > capacidad_disponible or cliente in excluidos_acumulados:
            continue
        seleccionados.append(cliente)
        capacidad_disponible -= d
        ahorro_acumulado += ahorro
        excluidos_acumulados |= incompatibles_cliente[cliente]

    if not seleccionados:
        return None

    seleccionados_por_origen: dict[int, int] = {}
    for cliente in seleccionados:
        origen = instalacion_asignada_al_cliente[cliente]
        seleccionados_por_origen[origen] = seleccionados_por_origen.get(origen, 0) + 1

    ahorro_por_cierres_origen = 0
    for origen, n_seleccionados_origen in seleccionados_por_origen.items():
        if n_seleccionados_origen == carga_por_instalacion[origen]:
            ahorro_por_cierres_origen += costo_fijo_instalacion[origen]

    delta = costo_fijo_instalacion[instalacion] - ahorro_acumulado - ahorro_por_cierres_origen
    return delta, seleccionados

def _abrir_instalaciones_rentables(
    instalacion_esta_abierta: list[bool],
    instalacion_asignada_al_cliente: list[int],
    capacidad_restante_instalacion: list[int],
    numero_instalaciones: int,
    numero_clientes: int,
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    incompatibles_cliente: list[set[int]] | None = None,
) -> tuple[SolucionCFLP, bool]:
    """N3 aplicado: abre instalaciones cerradas migrando a ellas los clientes
    con mayor ahorro de envío, si el ahorro acumulado (incluyendo el cierre
    incidental de instalaciones de origen que se vacían) supera el coste fijo
    de apertura. Usada por GVNS (vecindario N3 del VND); inverso de N2."""
    instalacion_esta_abierta = list(instalacion_esta_abierta)
    instalacion_asignada_al_cliente = list(instalacion_asignada_al_cliente)
    capacidad_restante_instalacion = list(capacidad_restante_instalacion)
    incompatibles = incompatibles_cliente or [set() for _ in range(numero_clientes)]

    # Ocupación de cada instalación, mantenida al día conforme se aplican
    # aperturas: evita recontar en O(n) con `any(...)` cada vez que hay que
    # comprobar si una instalación de origen se quedó sin clientes.
    carga_por_instalacion = [0] * numero_instalaciones
    for j in range(numero_clientes):
        carga_por_instalacion[instalacion_asignada_al_cliente[j]] += 1

    hubo_mejora = False

    for indice_instalacion_a_abrir in range(numero_instalaciones):
        if instalacion_esta_abierta[indice_instalacion_a_abrir]:
            continue

        propuesta = _evaluar_apertura_instalacion(
            indice_instalacion_a_abrir,
            instalacion_esta_abierta, instalacion_asignada_al_cliente,
            capacidad_restante_instalacion, numero_clientes, costo_fijo_instalacion,
            demanda_cliente, costo_envio, incompatibles, carga_por_instalacion,
        )
        if propuesta is None:
            continue
        delta, seleccionados = propuesta

        if delta >= 0:
            continue

        instalaciones_origen = {
            instalacion_asignada_al_cliente[indice_cliente] for indice_cliente in seleccionados
        }

        for indice_cliente in seleccionados:
            instalacion_origen = instalacion_asignada_al_cliente[indice_cliente]
            demanda = demanda_cliente[indice_cliente]
            capacidad_restante_instalacion[instalacion_origen] += demanda
            capacidad_restante_instalacion[indice_instalacion_a_abrir] -= demanda
            instalacion_asignada_al_cliente[indice_cliente] = indice_instalacion_a_abrir
            carga_por_instalacion[instalacion_origen] -= 1
            carga_por_instalacion[indice_instalacion_a_abrir] += 1
        instalacion_esta_abierta[indice_instalacion_a_abrir] = True

        for instalacion_origen in instalaciones_origen:
            if carga_por_instalacion[instalacion_origen] == 0:
                instalacion_esta_abierta[instalacion_origen] = False

        hubo_mejora = True

    return (
        instalacion_esta_abierta,
        instalacion_asignada_al_cliente,
        capacidad_restante_instalacion,
    ), hubo_mejora


## 2. GRASP

GRASP (Greedy Randomized Adaptive Search Procedure) construye en cada iteración
una solución greedy-aleatoria, controlada por el parámetro alfa, y la mejora con
búsqueda local hasta mínimo local. El bucle se repite N veces guardando la mejor.

### Estructuras de datos (GRASP)

In [341]:
class ResultadoGRASP(TypedDict):
    costo_total_minimo: float
    instalaciones_abiertas: list[bool]
    cliente_asignado_a_instalacion: list[int]
    capacidad_restante_por_instalacion: list[int]

### Fase 1: Construcción greedy aleatoria

`alfa` controla cuántas instalaciones entran en la Lista Restringida de Candidatos (RCL):
con `alfa=0` solo entra la más barata; con `alfa=1` entran todas las factibles.
El umbral es `costo_min + alfa*(costo_max - costo_min)`.
Los clientes se procesan en orden descendente de demanda para reducir infactibilidades.

In [342]:
def construir_solucion_greedy_aleatoria(
    numero_instalaciones: int,
    numero_clientes: int,
    capacidad_instalacion: list[int],
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    alfa: float,
    incompatibles_cliente: list[set[int]] | None = None,
) -> SolucionCFLP:
    """Solución greedy aleatoria: RCL con umbral alfa, clientes por demanda descendente."""
    instalacion_esta_abierta = [False] * numero_instalaciones
    instalacion_asignada_al_cliente = [-1] * numero_clientes
    capacidad_restante_instalacion = list(capacidad_instalacion)

    orden_clientes = sorted(
        range(numero_clientes),
        key=lambda j: demanda_cliente[j],
        reverse=True,
    )

    for indice_cliente in orden_clientes:

        dem = demanda_cliente[indice_cliente]
        inc = incompatibles_cliente[indice_cliente] if incompatibles_cliente else set()

        costos_m = []

        for indice_instalacion in range(numero_instalaciones):

            cabe = (
                capacidad_restante_instalacion[indice_instalacion]
                >= dem
            )

            if not cabe:
                costos_m.append(float("inf"))
                continue

            # Ningún cliente incompatible puede estar ya en esta instalación
            if any(
                instalacion_asignada_al_cliente[k] == indice_instalacion
                for k in inc
            ):
                costos_m.append(float("inf"))
                continue

            c_env = (
                costo_envio[indice_instalacion][indice_cliente]
                * dem
            )

            if instalacion_esta_abierta[indice_instalacion]:
                costo_marginal = c_env
            else:
                costo_marginal = (
                    c_env
                    + costo_fijo_instalacion[indice_instalacion]
                )

            costos_m.append(costo_marginal)

        factibles = [
            c for c in costos_m
            if c < float("inf")
        ]

        if not factibles:
            raise RuntimeError(
                f"No hay instalación factible para el cliente "
                f"{indice_cliente} (capacidad o incompatibilidades). "
                f"La instancia podría ser infactible."
            )

        c_min = min(factibles)
        c_max = max(factibles)

        umbral = (
            c_min
            + alfa * (c_max - c_min)
        )

        rcl = [
            i for i in range(numero_instalaciones)
            if costos_m[i] <= umbral
        ]

        instalacion_elegida = random.choice(rcl)

        instalacion_asignada_al_cliente[indice_cliente] = instalacion_elegida
        capacidad_restante_instalacion[instalacion_elegida] -= (
            dem
        )
        instalacion_esta_abierta[instalacion_elegida] = True

    return (
        instalacion_esta_abierta,
        instalacion_asignada_al_cliente,
        capacidad_restante_instalacion,
    )

### Búsqueda local completa

In [343]:
def mejorar_solucion_con_busqueda_local(
    instalacion_esta_abierta: list[bool],
    instalacion_asignada_al_cliente: list[int],
    capacidad_restante_instalacion: list[int],
    numero_instalaciones: int,
    numero_clientes: int,
    capacidad_instalacion: list[int],
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    incompatibles_cliente: list[set[int]] | None = None,
) -> SolucionCFLP:
    """Ciclos de 1-shift + cierre + apertura de instalaciones (N1+N2+N3) hasta
    alcanzar un mínimo local. Sin N3, la búsqueda solo puede reasignar entre
    instalaciones ya abiertas (N1) o cerrar (N2), nunca abrir una nueva por su
    cuenta — el mismo problema estructural que el 1-shift inerte del capítulo
    de SA. Probado empíricamente: añadir N3 reduce el gap medio de GRASP base
    de +26--43% a +11--15% en wlp01--wlp04, consistente en todas las semillas."""
    sol: SolucionCFLP = (
        list(instalacion_esta_abierta),
        list(instalacion_asignada_al_cliente),
        list(capacidad_restante_instalacion),
    )

    while True:
        sol, mejora_mov1 = _reasignar_clientes_en_barrido(
            sol[0], sol[1], sol[2],
            numero_instalaciones, numero_clientes,
            costo_fijo_instalacion, demanda_cliente, costo_envio,
            incompatibles_cliente,
        )
        sol, mejora_mov2 = _cerrar_instalaciones_poco_rentables(
            sol[0], sol[1], sol[2],
            numero_instalaciones, numero_clientes,
            costo_fijo_instalacion, demanda_cliente, costo_envio,
            incompatibles_cliente,
        )
        sol, mejora_mov3 = _abrir_instalaciones_rentables(
            sol[0], sol[1], sol[2],
            numero_instalaciones, numero_clientes,
            costo_fijo_instalacion, demanda_cliente, costo_envio,
            incompatibles_cliente,
        )
        if not mejora_mov1 and not mejora_mov2 and not mejora_mov3:
            break

    return sol

### Ejecución GRASP

Bucle de N iteraciones: construir -> mejorar con BL -> guardar si es mejor.
Si usar_alfa_adaptativo=True, el selector ML actualiza alfa tras cada iteración,
usando las primeras muestras_exploracion como exploración aleatoria.

In [344]:
def ejecutar_grasp(
    numero_instalaciones: int,
    numero_clientes: int,
    capacidad_instalacion: list[int],
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    alfa: float = 0.5,
    numero_maximo_de_iteraciones: int = 100,
    semilla_aleatoria: int = 0,
    usar_alfa_adaptativo: bool = True,
    muestras_exploracion: int = 15,
    pares_incompatibles: list[tuple[int, int]] | None = None,
    tiempo_limite_s: float | None = None,
) -> ResultadoGRASP:
    """GRASP con alfa fijo o adaptativo (ML). Devuelve la mejor solución tras N
    iteraciones, o hasta agotar `tiempo_limite_s` si se indica (lo que ocurra
    primero)."""
    random.seed(semilla_aleatoria)
    t_inicio = time.perf_counter()

    incompatibles = _precomputar_incompatibles(numero_clientes, pares_incompatibles or [])

    mejor_costo_encontrado: float = float("inf")
    mejor_instalacion_esta_abierta: list[bool] | None = None
    mejor_instalacion_asignada: list[int] | None = None
    mejor_capacidad_restante: list[int] | None = None

    if usar_alfa_adaptativo:
        selector_ml = SelectorAlfaML(min_muestras=muestras_exploracion)

    print("=" * 65)
    print("  GRASP - MS-CFLP-CI")
    print("=" * 65)
    print(f"  Instalaciones      : {numero_instalaciones}")
    print(f"  Clientes           : {numero_clientes}")
    print(f"  Pares incompatibles: {len(pares_incompatibles or [])}")
    if usar_alfa_adaptativo:
        print("  Alfa               : adaptativo (regresión lineal ML)")
        print(f"  Exploración inicial: {muestras_exploracion} iteraciones")
    else:
        print(f"  Alfa (aleatoriedad): {alfa}")
    print(f"  Iteraciones GRASP  : {numero_maximo_de_iteraciones}")
    print("=" * 65)

    for iteracion in range(1, numero_maximo_de_iteraciones + 1):
        if tiempo_limite_s is not None and time.perf_counter() - t_inicio >= tiempo_limite_s:
            break

        alfa_esta_iteracion = (
            selector_ml.seleccionar_alfa() if usar_alfa_adaptativo else alfa
        )

        (
            instalacion_esta_abierta,
            instalacion_asignada_al_cliente,
            capacidad_restante_instalacion,
        ) = construir_solucion_greedy_aleatoria(
            numero_instalaciones,
            numero_clientes,
            capacidad_instalacion,
            costo_fijo_instalacion,
            demanda_cliente,
            costo_envio,
            alfa_esta_iteracion,
            incompatibles,
        )

        costo_construida = calcular_costo_total(
            instalacion_esta_abierta,
            instalacion_asignada_al_cliente,
            costo_fijo_instalacion,
            demanda_cliente,
            costo_envio,
        )

        (
            instalacion_esta_abierta,
            instalacion_asignada_al_cliente,
            capacidad_restante_instalacion,
        ) = mejorar_solucion_con_busqueda_local(
            instalacion_esta_abierta,
            instalacion_asignada_al_cliente,
            capacidad_restante_instalacion,
            numero_instalaciones,
            numero_clientes,
            capacidad_instalacion,
            costo_fijo_instalacion,
            demanda_cliente,
            costo_envio,
            incompatibles,
        )

        costo_mejorada = calcular_costo_total(
            instalacion_esta_abierta,
            instalacion_asignada_al_cliente,
            costo_fijo_instalacion,
            demanda_cliente,
            costo_envio,
        )

        if usar_alfa_adaptativo:
            selector_ml.registrar(alfa_esta_iteracion, costo_mejorada)

        if costo_mejorada < mejor_costo_encontrado:
            mejor_costo_encontrado = costo_mejorada
            mejor_instalacion_esta_abierta = list(instalacion_esta_abierta)
            mejor_instalacion_asignada = list(instalacion_asignada_al_cliente)
            mejor_capacidad_restante = list(capacidad_restante_instalacion)
            indicador_nueva_mejor = " << NUEVA MEJOR"
        else:
            indicador_nueva_mejor = ""

        indicador_alfa = (
            f"alfa={alfa_esta_iteracion:.2f}"
            if usar_alfa_adaptativo
            else f"alfa={alfa:.2f}"
        )
        print(
            f"  Iter {iteracion:>4} | "
            f"{indicador_alfa} | "
            f"Construida: {costo_construida:>8,.0f} | "
            f"Mejorada: {costo_mejorada:>8,.0f} | "
            f"Mejor: {mejor_costo_encontrado:>8,.0f}"
            f"{indicador_nueva_mejor}"
        )

    if usar_alfa_adaptativo:
        alfas_usados, costos_registrados, modelo_final = selector_ml.obtener_estado()
        if modelo_final is not None:
            coefs = modelo_final.coef_
            print(
                f"\n  [ML] Modelo entrenado con "
                f"{len(alfas_usados)} observaciones."
            )
            print(
                f"  [ML] Coeficientes (grado 2): "
                f"{[round(c, 4) for c in coefs]}"
            )
            alfa_optimo = alfas_usados[int(np.argmin(costos_registrados))]
            print(f"  [ML] Mejor alfa observado: {alfa_optimo:.2f}")

    print("=" * 65)
    print(f"  SOLUCIÓN FINAL - Costo total: {mejor_costo_encontrado:,.0f}")
    print("=" * 65)

    return {
        "costo_total_minimo": mejor_costo_encontrado,
        "instalaciones_abiertas": mejor_instalacion_esta_abierta,
        "cliente_asignado_a_instalacion": mejor_instalacion_asignada,
        "capacidad_restante_por_instalacion": mejor_capacidad_restante,
    }

### Mostrar resultados detallados

In [345]:
def mostrar_resultados_detallados(
    resultado_grasp: ResultadoGRASP,
    capacidad_instalacion: list[int],
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
) -> None:
    """Imprime instalaciones abiertas, asignaciones y desglose de costos."""
    instalaciones_abiertas = resultado_grasp["instalaciones_abiertas"]
    cliente_asignado = resultado_grasp["cliente_asignado_a_instalacion"]
    capacidad_restante = resultado_grasp[
        "capacidad_restante_por_instalacion"
    ]
    costo_total_minimo = resultado_grasp["costo_total_minimo"]

    numero_instalaciones = len(instalaciones_abiertas)
    numero_clientes = len(cliente_asignado)

    print("\n" + "=" * 65)
    print("  DETALLE DE INSTALACIONES ABIERTAS")
    print("=" * 65)

    suma_costos_fijos = 0
    suma_costos_de_envio = 0

    for i in range(numero_instalaciones):

        if not instalaciones_abiertas[i]:
            continue

        clientes_de_esta = [
            j for j in range(numero_clientes)
            if cliente_asignado[j] == i
        ]

        capacidad_usada = (
            capacidad_instalacion[i] - capacidad_restante[i]
        )

        costo_fijo_de_esta = costo_fijo_instalacion[i]
        suma_costos_fijos += costo_fijo_de_esta

        costo_envio_de_esta = sum(
            demanda_cliente[j] * costo_envio[i][j]
            for j in clientes_de_esta
        )
        suma_costos_de_envio += costo_envio_de_esta

        print(
            f"  Instalación {i:>3} | "
            f"Costo fijo: {costo_fijo_de_esta:>4} | "
            f"Capacidad: {capacidad_usada:>3}/{capacidad_instalacion[i]:>3}"
            f" | Clientes: {len(clientes_de_esta):>3} | "
            f"Costo envíos: {costo_envio_de_esta:>7,.0f}"
        )

    print("-" * 65)
    print(f"  Total instalaciones abiertas : {sum(instalaciones_abiertas)}")
    print(f"  Costo fijo total             : {suma_costos_fijos:>10,.0f}")
    print(f"  Costo de envíos total        : {suma_costos_de_envio:>10,.0f}")
    print(f"  COSTO TOTAL                  : {costo_total_minimo:>10,.0f}")
    print("=" * 65)


### GRASP + ML: selector adaptativo de alfa (regresión polinómica)

Ajusta un polinomio de grado configurable sobre el historial de pares (alfa, coste normalizado)
observados en iteraciones anteriores y elige el alfa que minimiza el coste predicho.
El grado se selecciona empíricamente antes de ejecutar las comparativas (véase la celda de comparación de grados).

**Diseño en dos fases** dentro del mismo presupuesto de *N* iteraciones:

1. **Exploración** (primeras `min_muestras` iteraciones): alfa aleatorio uniforme en [0, 1],
   sin modelo, para obtener puntos de entrenamiento distribuidos.
2. **Explotación** (iteraciones restantes): elige el alfa con menor coste predicho
   por el polinomio.


In [346]:
from scipy.optimize import minimize_scalar

class SelectorAlfaML:
    """Selección de alfa con regresión polinómica (grado configurable), minimización continua sobre [0.01, 0.99]."""

    def __init__(
        self, grado: int, min_muestras: int = 20
    ) -> None:
        self.min_muestras = min_muestras
        self.grado = grado
        self.historial_alfa: list[float] = []
        self.historial_costo: list[float] = []
        self.poly = PolynomialFeatures(degree=grado, include_bias=False)
        self.modelo_actual: LinearRegression | None = None

    def registrar(self, alfa: float, costo: float) -> None:
        self.historial_alfa.append(alfa)
        self.historial_costo.append(costo)

    def _entrenar_modelo(self, X, y) -> LinearRegression:
        return LinearRegression().fit(X, y)

    def _predecir(self, modelo: LinearRegression, X_cand) -> np.ndarray:
        return modelo.predict(X_cand)

    def seleccionar_alfa(self) -> float:
        n = len(self.historial_alfa)

        # Fase de exploración
        if n < self.min_muestras:
            return round(random.uniform(0.00, 1.00), 4)

        # Fase de explotación: entrenar modelo
        X = self.poly.fit_transform(
            np.array(self.historial_alfa, dtype=float).reshape(-1, 1)
        )
        costos = np.array(self.historial_costo, dtype=float)
        cmin, cmax = costos.min(), costos.max()
        y_norm = (costos - cmin) / (cmax - cmin) if cmax > cmin else np.zeros_like(costos)

        modelo = self._entrenar_modelo(X, y_norm)
        self.modelo_actual = modelo

        # Minimización continua de f_hat sobre [0.01, 0.99] (método de Brent acotado)
        def _objetivo(alfa: float) -> float:
            X_a = self.poly.transform(np.array([[alfa]]))
            return float(self._predecir(modelo, X_a)[0])

        resultado = minimize_scalar(_objetivo, bounds=(0.01, 0.99), method="bounded")
        return round(float(resultado.x), 4)

    def obtener_estado(
        self,
    ) -> tuple[list[float], list[float], LinearRegression | None]:
        return (
            list(self.historial_alfa),
            list(self.historial_costo),
            self.modelo_actual,
        )


In [347]:
class SelectorAlfaMLTemporizador(SelectorAlfaML):
    """SelectorAlfaML que registra tiempos de entrenamiento e inferencia."""

    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.tiempo_entrenamiento_s: float = 0.0
        self.tiempo_inferencia_s: float = 0.0

    def _entrenar_modelo(self, X, y) -> LinearRegression:
        t0 = time.perf_counter()
        modelo = super()._entrenar_modelo(X, y)
        self.tiempo_entrenamiento_s += time.perf_counter() - t0
        return modelo

    def _predecir(self, modelo: LinearRegression, X_cand) -> np.ndarray:
        t0 = time.perf_counter()
        pred = super()._predecir(modelo, X_cand)
        self.tiempo_inferencia_s += time.perf_counter() - t0
        return pred


def _suprimir_stdout(fn, *args, **kwargs):
    """Llama fn suprimiendo cualquier salida a stdout."""
    _stdout = sys.stdout
    sys.stdout = io.StringIO()
    try:
        result = fn(*args, **kwargs)
    finally:
        sys.stdout = _stdout
    return result



### GRASP + ML: ejecución con selector polinómico

Bucle de `n_iter` iteraciones con el mismo presupuesto que GRASP base.
El selector polinómico sustituye el alfa fijo en dos fases:

- **Exploración** (`rand`): las primeras `min_muestras` iteraciones seleccionan $\alpha$ uniformemente al azar para acumular el historial $(\alpha_t, C_t)$ mínimo que necesita el ajuste del modelo.
- **Explotación** (`ML`): a partir de la muestra `min_muestras`, ajusta la regresión polinómica sobre el historial y selecciona el $\alpha$ que minimiza $\hat{f}$ en $[0.01, 0.99]$ mediante minimización continua (método de Brent acotado).

> El grado del polinomio **no está fijado aquí**: debe pasarse como argumento `grado` y en las comparativas se usa `_MEJOR_GRADO`, el grado que minimizó el gap medio en la comparación de grados previa.
> `min_muestras=15` garantiza un ajuste mínimamente estable antes de explotar el modelo; por tanto `n_iter` debe ser al menos `min_muestras + 1`.

In [348]:
def ejecutar_grasp_ml(
    datos: DatosCFLP,
    grado: int,
    n_iter: int = 50,
    min_muestras: int = 15,
    semilla: int = 0,
    verbose: bool = True,
    tiempo_limite_s: float | None = None,
) -> "ResultadoBenchmark":
    """GRASP con selector adaptativo de alfa (regresión polinómica)."""
    random.seed(semilla)
    incompatibles = _precomputar_incompatibles(
        datos["numero_clientes"], datos["pares_incompatibles"]
    )
    selector = SelectorAlfaML(min_muestras=min_muestras, grado=grado)
    mejor_costo = float("inf")
    t_train = 0.0
    t_infer = 0.0
    t_inicio = time.perf_counter()

    if verbose:
        print("=" * 60)
        print(f"  GRASP + ML  (regresión polinómica grado {grado})")
        print("=" * 60)
        print(f"  {'Iter':>4}  {'Fase':<6}  {'Alfa':>6}  {'Coste':>12}  {'Mejor':>12}")
        print(f"  {'-'*50}")

    for it in range(1, n_iter + 1):
        if tiempo_limite_s is not None and time.perf_counter() - t_inicio >= tiempo_limite_s:
            break

        _t0 = time.perf_counter()
        alfa = selector.seleccionar_alfa()
        t_infer += time.perf_counter() - _t0

        fase = "rand" if len(selector.historial_alfa) < min_muestras else "ML"

        ab, asig, rest = construir_solucion_greedy_aleatoria(
            datos["numero_instalaciones"], datos["numero_clientes"],
            datos["capacidad_instalacion"], datos["costo_fijo_instalacion"],
            datos["demanda_cliente"], datos["costo_envio"], alfa, incompatibles,
        )
        ab, asig, rest = mejorar_solucion_con_busqueda_local(
            ab, asig, rest,
            datos["numero_instalaciones"], datos["numero_clientes"],
            datos["capacidad_instalacion"], datos["costo_fijo_instalacion"],
            datos["demanda_cliente"], datos["costo_envio"], incompatibles,
        )
        costo = calcular_costo_total(
            ab, asig, datos["costo_fijo_instalacion"],
            datos["demanda_cliente"], datos["costo_envio"],
        )

        _t0 = time.perf_counter()
        selector.registrar(alfa, costo)
        t_train += time.perf_counter() - _t0

        if costo < mejor_costo:
            mejor_costo = costo

        if verbose:
            print(f"  {it:>4}  {fase:<6}  {alfa:>6.2f}  {costo:>12,.0f}  {mejor_costo:>12,.0f}")

    t_total = time.perf_counter() - t_inicio
    if verbose:
        print(f"  {'-'*50}")
        print(f"  Mejor coste: {mejor_costo:,.0f}")
        print(f"  Tiempo total: {t_total:.2f}s"
              f"  (ML entreno: {t_train:.3f}s  inferencia: {t_infer:.3f}s)")

    return ResultadoBenchmark(
        nombre=f"GRASP+ML(g{grado})",
        costo_final=mejor_costo,
        tiempo_total_s=t_total,
        tiempo_entrenamiento_ml_s=t_train,
        tiempo_inferencia_ml_s=t_infer,
        modelo_ml=selector,
    )

### Evaluación empírica de los selectores de alfa

Esta celda justifica empíricamente la elección del grado polinómico para GRASP+ML
evaluando varios candidatos sobre las instancias `wlp01`–`wlp04`.

#### Qué se mide

Para cada instancia se generan **N = 20 pares (alfa, coste)** ejecutando GRASP + BL
con alfas aleatorios. Sobre esas muestras se calculan:

- **R² LOO** (*leave-one-out*): mide cuánta variación de coste entre distintos alfas
  explica el modelo. Se usa LOO en lugar de R² sobre entrenamiento para penalizar el
  sobreajuste. R² > 0 indica que el modelo genera predicciones útiles.
- **RMSE y MAE LOO**: error absoluto de predicción en las mismas unidades que el coste.
- **|Δα*|**: diferencia entre el alfa óptimo observado y el predicho por el modelo.


In [349]:
import time as _time
import os
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

INSTANCIAS_EVAL = ["wlp01.dzn", "wlp02.dzn", "wlp03.dzn", "wlp04.dzn"]
DATA_DIR        = "Instaces_MS-CFLP-CI/Public"
N_MUESTRAS      = 20   # iteraciones GRASP con alfa aleatorio por instancia
SEMILLA         = 42

# ── Ejecuta GRASP+BL con un alfa fijo ──────────────────────────────────────
def _grasp_con_alfa(datos, alfa, semilla_local=0):
    incompatibles = _precomputar_incompatibles(
        datos["numero_clientes"], datos["pares_incompatibles"]
    )
    random.seed(semilla_local)
    ab, asig, rest = construir_solucion_greedy_aleatoria(
        datos["numero_instalaciones"], datos["numero_clientes"],
        datos["capacidad_instalacion"], datos["costo_fijo_instalacion"],
        datos["demanda_cliente"], datos["costo_envio"], alfa, incompatibles,
    )
    ab, asig, rest = mejorar_solucion_con_busqueda_local(
        ab, asig, rest,
        datos["numero_instalaciones"], datos["numero_clientes"],
        datos["capacidad_instalacion"], datos["costo_fijo_instalacion"],
        datos["demanda_cliente"], datos["costo_envio"], incompatibles,
    )
    return calcular_costo_total(
        ab, asig, datos["costo_fijo_instalacion"],
        datos["demanda_cliente"], datos["costo_envio"],
    )

# ── Definición de modelos ───────────────────────────────────────────────────
MODELOS = {
    "Poly-1": ("poly", 1),
    "Poly-2": ("poly", 2),
    "Poly-3": ("poly", 3),
}

def _ajustar_y_predecir(nombre, X_tr, y_tr, X_te):
    kind, param = MODELOS[nombre]
    if kind == "poly":
        pf  = PolynomialFeatures(param, include_bias=True)
        reg = LinearRegression()
        reg.fit(pf.fit_transform(X_tr), y_tr)
        return reg.predict(pf.transform(X_te))

def _loo_cv(nombre, X, y):
    kind, param = MODELOS[nombre]
    n = len(X)
    # Poly-d necesita al menos d+2 puntos para que LOO tenga un grado de libertad
    if kind == "poly" and n < param + 2:
        return np.nan, np.nan, np.nan
    loo    = LeaveOneOut()
    y_pred = np.empty(n)
    for tr_idx, te_idx in loo.split(X):
        try:
            y_pred[te_idx] = _ajustar_y_predecir(nombre, X[tr_idx], y[tr_idx], X[te_idx])
        except Exception:
            y_pred[te_idx] = np.nan
    mask = ~np.isnan(y_pred)
    if mask.sum() < 2:
        return np.nan, np.nan, np.nan
    r2   = r2_score(y[mask], y_pred[mask])
    rmse = np.sqrt(mean_squared_error(y[mask], y_pred[mask]))
    mae  = mean_absolute_error(y[mask], y_pred[mask])
    return r2, rmse, mae

def _alfa_optima_pred(nombre, X_all, y_all):
    candidatos = np.linspace(0.0, 1.0, 200).reshape(-1, 1)
    preds = _ajustar_y_predecir(nombre, X_all, y_all, candidatos)
    return float(candidatos[np.argmin(preds)])

_EJECUTAR_EVALUACION_LOO = False  # True → LOO CV + learning curve + plot (~1 min)

if _EJECUTAR_EVALUACION_LOO:
    # ── Curva de aprendizaje: tamaños de muestra ───────────────────────────────
    TAMANIOS_LC = [3, 5, 7, 10, 15, 20]
    
    # ── Bucle principal por instancia ───────────────────────────────────────────
    metricas_global = {nm: {"r2": [], "rmse": [], "mae": []} for nm in MODELOS}
    lc_global       = {nm: {k: [] for k in TAMANIOS_LC} for nm in MODELOS}
    
    for archivo in INSTANCIAS_EVAL:
        nombre_inst = archivo.replace(".dzn", "")
        datos = leer_instancia_dzn(os.path.join(DATA_DIR, archivo))
        F, C  = datos["numero_instalaciones"], datos["numero_clientes"]
    
        t0 = _time.perf_counter()
        np.random.seed(SEMILLA); random.seed(SEMILLA)
        alfas  = np.array([round(random.uniform(0.0, 1.0), 2) for _ in range(N_MUESTRAS)])
        costos = np.array([_grasp_con_alfa(datos, a, SEMILLA + i) for i, a in enumerate(alfas)])
        t_gen  = _time.perf_counter() - t0
    
        X = alfas.reshape(-1, 1)
        y = costos
        alfa_obs = alfas[np.argmin(costos)]
    
        print(f"\n{'='*62}")
        print(f"  {nombre_inst}  ({F} almacenes, {C} tiendas)  — muestras en {t_gen:.1f}s")
        print(f"  α* observado = {alfa_obs:.2f}  |  coste mín observado = {costos.min():.0f}")
        print(f"\n  {'Modelo':<9} {'R²(LOO)':>9} {'RMSE(LOO)':>11} {'MAE(LOO)':>10}"
              f" {'α* pred':>8} {'|Δα*|':>6}")
        print(f"  {'-'*57}")
    
        for nm in MODELOS:
            r2, rmse, mae = _loo_cv(nm, X, y)
            if np.isnan(r2):
                print(f"  {nm:<9} {'N/A':>9} {'N/A':>11} {'N/A':>10}")
                continue
            alfa_pred  = _alfa_optima_pred(nm, X, y)
            delta_alfa = abs(alfa_pred - alfa_obs)
            print(f"  {nm:<9} {r2:>+9.4f} {rmse:>11.0f} {mae:>10.0f}"
                  f" {alfa_pred:>8.2f} {delta_alfa:>6.2f}")
            metricas_global[nm]["r2"].append(r2)
            metricas_global[nm]["rmse"].append(rmse)
            metricas_global[nm]["mae"].append(mae)
    
        # Curva de aprendizaje
        for nm in MODELOS:
            for k in TAMANIOS_LC:
                r2k, _, _ = _loo_cv(nm, X[:k], y[:k])
                lc_global[nm][k].append(r2k)
    
    # ── Resumen global ──────────────────────────────────────────────────────────
    print(f"\n\n{'='*62}")
    print("  RESUMEN GLOBAL — media sobre wlp01–wlp04")
    print(f"  {'Modelo':<9} {'R²(LOO)':>9} {'RMSE(LOO)':>11} {'MAE(LOO)':>10}")
    print(f"  {'-'*41}")
    for nm in MODELOS:
        vals_r2   = metricas_global[nm]["r2"]
        vals_rmse = metricas_global[nm]["rmse"]
        vals_mae  = metricas_global[nm]["mae"]
        if not vals_r2:
            continue
        print(f"  {nm:<9} {np.mean(vals_r2):>+9.4f}"
              f" {np.mean(vals_rmse):>11.0f} {np.mean(vals_mae):>10.0f}")
    
    # ── Curva de aprendizaje ────────────────────────────────────────────────────
    print(f"\n\n  Curva de aprendizaje — R²(LOO) medio (wlp01–wlp04):")
    header_lc = f"  {'n':>4}"
    for nm in MODELOS:
        header_lc += f"  {nm:>9}"
    print(header_lc)
    print(f"  {'-'*(6 + len(MODELOS) * 11)}")
    for k in TAMANIOS_LC:
        row = f"  {k:>4}"
        for nm in MODELOS:
            vals = [v for v in lc_global[nm][k] if v is not None and not np.isnan(v)]
            row += f"  {np.mean(vals):>+9.4f}" if vals else f"  {'N/A':>9}"
        print(row)
    
    print()
    print("  Nota: Poly-d requiere ≥ d+2 muestras para LOO significativo.")
    print("  GRASP+ML usa n=3–5 muestras antes de explotar el modelo Poly-2.")
    
    # ── Gráfico: curva alfa→coste + ajustes (primera instancia) ────────────────
    fig, axes = plt.subplots(1, len(INSTANCIAS_EVAL), figsize=(14, 4), sharey=False)
    colores = {"Poly-1": "tab:blue", "Poly-2": "tab:orange",
               "Poly-3": "tab:green", "RF-50": "tab:red"}
    
    for ax, archivo in zip(axes, INSTANCIAS_EVAL):
        nombre_inst = archivo.replace(".dzn", "")
        datos = leer_instancia_dzn(os.path.join(DATA_DIR, archivo))
        np.random.seed(SEMILLA); random.seed(SEMILLA)
        alfas_p  = np.array([round(random.uniform(0.0, 1.0), 2) for _ in range(N_MUESTRAS)])
        costos_p = np.array([_grasp_con_alfa(datos, a, SEMILLA + i) for i, a in enumerate(alfas_p)])
        X_p = alfas_p.reshape(-1, 1)
    
        ax.scatter(alfas_p, costos_p, color="gray", s=20, alpha=0.6, label="muestras")
        grid = np.linspace(0.0, 1.0, 200).reshape(-1, 1)
        for nm, col in colores.items():
            try:
                preds = _ajustar_y_predecir(nm, X_p, costos_p, grid)
                ax.plot(grid, preds, color=col, linewidth=1.5, label=nm)
            except Exception:
                pass
        ax.set_title(nombre_inst)
        ax.set_xlabel("alfa (α)")
        ax.set_ylabel("coste")
        ax.legend(fontsize=7)
    
    plt.suptitle("Ajuste de modelos: curva alfa → coste con BL (20 muestras por instancia)")
    plt.tight_layout()
    plt.show()


### Comparación de grados polinómicos en GRASP+ML

Determina empíricamente el grado óptimo del polinomio del selector de $\alpha$.
Se evalúan los grados $d \in \{1, 2, 3, 4, 5, 6\}$ ejecutando `ejecutar_grasp_ml` sobre `wlp01`–`wlp05` con varias semillas y se elige el grado que produce el menor **gap medio** respecto a la referencia.

El resultado (`_MEJOR_GRADO`) se usa en todas las comparativas posteriores.

> Esta celda debe ejecutarse **antes** que cualquier comparativa de variantes GRASP, ya que `_MEJOR_GRADO` es una dependencia de `grasp-compare-code`.

In [350]:
import os as _os2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

_EJECUTAR_COMPARATIVA = False   # True → corre la comparativa completa (~46 min)
                                # False → usa _MEJOR_GRADO_DEFAULT directamente
_MEJOR_GRADO_DEFAULT  = 2      # grado usado cuando _EJECUTAR_COMPARATIVA = False

_INSTANCIAS_GRADO = ["wlp01.dzn", "wlp02.dzn", "wlp03.dzn", "wlp04.dzn"]
_REFERENCIA_GRADO = {"wlp01": 28716, "wlp02": 52952, "wlp03": 64296, "wlp04": 84633}
_GRADOS     = [1, 2, 3, 4, 5, 6]
_N_ITER_G   = 50
_MIN_M_G    = 15
_SEMILLAS_G = [0, 1, 2, 3, 4]

if not _EJECUTAR_COMPARATIVA:
    _MEJOR_GRADO = _MEJOR_GRADO_DEFAULT
    print(f"  [comparativa omitida] _MEJOR_GRADO = {_MEJOR_GRADO} (por defecto)")
else:
    _total = len(_SEMILLAS_G) * len(_INSTANCIAS_GRADO) * len(_GRADOS)
    _done  = 0

    _res = {
        (arch.replace(".dzn", ""), g): {"gaps": [], "tiempos": []}
        for arch in _INSTANCIAS_GRADO
        for g in _GRADOS
    }

    for _sem in _SEMILLAS_G:
        print(f"\n  Semilla {_sem} / {_SEMILLAS_G[-1]}")
        for archivo in _INSTANCIAS_GRADO:
            nombre = archivo.replace(".dzn", "")
            datos  = leer_instancia_dzn(_os2.path.join("Instaces_MS-CFLP-CI/Public", archivo))
            ref    = _REFERENCIA_GRADO[nombre]
            for g in _GRADOS:
                _done += 1
                pct = 100 * _done / _total
                print(f"    [{pct:5.1f}%] {nombre}  Poly-{g}...", end=" ", flush=True)
                rb  = ejecutar_grasp_ml(datos, grado=g, n_iter=_N_ITER_G,
                                        min_muestras=_MIN_M_G, semilla=_sem, verbose=False)
                gap = 100 * (rb.costo_final - ref) / ref
                _res[(nombre, g)]["gaps"].append(gap)
                _res[(nombre, g)]["tiempos"].append(rb.tiempo_total_s)
                print(f"gap {gap:+.2f}%  ({rb.tiempo_total_s:.1f}s)")

    # ── Tabla (media ± std) ──────────────────────────────────────────────────────
    instancias = [a.replace(".dzn", "") for a in _INSTANCIAS_GRADO]
    print()
    print(f"  {'Instancia':<10}", end="")
    for g in _GRADOS:
        print(f" │ {'Poly-'+str(g):^21}", end="")
    print()
    print(f"  {'':10}", end="")
    for g in _GRADOS:
        print(f" │ {'Gap% (media±std)':>14} {'t(s)':>5}", end="")
    print()
    print(f"  {'-'*10}", end="")
    for _ in _GRADOS:
        print(f" ┼ {'-'*14} {'-'*5}", end="")
    print()

    avg_por_grado = {}
    for nombre in instancias:
        print(f"  {nombre:<10}", end="")
        for g in _GRADOS:
            gaps   = _res[(nombre, g)]["gaps"]
            mu_gap = np.mean(gaps);  sd_gap = np.std(gaps)
            mu_t   = np.mean(_res[(nombre, g)]["tiempos"])
            print(f" │ {mu_gap:>+6.2f}±{sd_gap:<5.2f}% {mu_t:>4.1f}s", end="")
        print()

    print(f"  {'Media':<10}", end="")
    for g in _GRADOS:
        all_gaps = [gap for n in instancias for gap in _res[(n, g)]["gaps"]]
        avg = np.mean(all_gaps)
        avg_por_grado[g] = avg
        print(f" │ {avg:>+6.2f}±{'':5}  {'':5}", end="")
    print()

    _MEJOR_GRADO = min(avg_por_grado, key=avg_por_grado.get)
    print(f"\n  Mejor grado: Poly-{_MEJOR_GRADO} (gap medio {avg_por_grado[_MEJOR_GRADO]:+.2f}%)")
    print(f"  → Se usará Poly-{_MEJOR_GRADO} en las comparaciones posteriores")
    print(f"  Semillas: {_SEMILLAS_G}  |  {_N_ITER_G} iter  |  {_MIN_M_G} min_muestras")

    # ── Gráficas ─────────────────────────────────────────────────────────────────
    _cmap   = plt.get_cmap("tab10")
    _colors = {g: _cmap(i) for i, g in enumerate(_GRADOS)}

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(
        f"Comparación de grados polinómicos · {len(_SEMILLAS_G)} semillas · {_N_ITER_G} iter ({_MIN_M_G} expl + {_N_ITER_G - _MIN_M_G} exploit)",
        fontsize=13, fontweight="bold",
    )

    # — 1. Box plot —
    bp_data = [[gap for n in instancias for gap in _res[(n, g)]["gaps"]] for g in _GRADOS]
    bp = axes[0].boxplot(bp_data, patch_artist=True,
                         medianprops={"color": "black", "linewidth": 1.8})
    for patch, g in zip(bp["boxes"], _GRADOS):
        patch.set_facecolor(_colors[g])
        patch.set_alpha(0.85)
        if g == _MEJOR_GRADO:
            patch.set_linewidth(2.5)
            patch.set_edgecolor("black")
    axes[0].set_xticks(range(1, len(_GRADOS) + 1))
    axes[0].set_xticklabels([f"Poly-{g}" for g in _GRADOS])
    axes[0].set_title("Distribución de gap% por grado\n(todas las instancias × semillas)")
    axes[0].set_ylabel("Gap (%)")
    axes[0].axhline(avg_por_grado[_MEJOR_GRADO], color="red", linewidth=1,
                    linestyle="--", alpha=0.6, label=f"Media mínima (Poly-{_MEJOR_GRADO})")
    axes[0].legend(fontsize=8)

    # — 2. Líneas por instancia —
    for nombre in instancias:
        mu_gaps = [np.mean(_res[(nombre, g)]["gaps"]) for g in _GRADOS]
        axes[1].plot(_GRADOS, mu_gaps, marker="o", label=nombre, linewidth=1.8)
    mu_global = [avg_por_grado[g] for g in _GRADOS]
    axes[1].plot(_GRADOS, mu_global, marker="D", color="black", linewidth=2.5,
                 linestyle="--", label="Media global", zorder=5)
    axes[1].axvline(_MEJOR_GRADO, color="red", linewidth=1, linestyle=":", alpha=0.7)
    axes[1].set_xticks(_GRADOS)
    axes[1].set_xticklabels([f"Poly-{g}" for g in _GRADOS])
    axes[1].set_title("Gap% medio por grado e instancia")
    axes[1].set_ylabel("Gap medio (%)")
    axes[1].set_xlabel("Grado del polinomio")
    axes[1].legend(fontsize=8)

    # — 3. Barras con error —
    mu_g  = [np.mean([gap for n in instancias for gap in _res[(n, g)]["gaps"]]) for g in _GRADOS]
    sd_g  = [np.std( [gap for n in instancias for gap in _res[(n, g)]["gaps"]]) for g in _GRADOS]
    bars = axes[2].bar(_GRADOS, mu_g, yerr=sd_g, capsize=5,
                       color=[_colors[g] for g in _GRADOS], alpha=0.88, width=0.6)
    for bar, g in zip(bars, _GRADOS):
        if g == _MEJOR_GRADO:
            bar.set_edgecolor("black")
            bar.set_linewidth(2.5)
    axes[2].set_xticks(_GRADOS)
    axes[2].set_xticklabels([f"Poly-{g}" for g in _GRADOS])
    axes[2].set_title("Gap% medio global por grado\n(± std entre instancias × semillas)")
    axes[2].set_ylabel("Gap medio (%)")
    axes[2].set_xlabel("Grado del polinomio")
    best_val = avg_por_grado[_MEJOR_GRADO]
    axes[2].annotate(f"mejor\nPoly-{_MEJOR_GRADO}\n{best_val:+.2f}%",
                     xy=(_MEJOR_GRADO, best_val),
                     xytext=(_MEJOR_GRADO + 0.6, best_val + max(sd_g) * 0.5),
                     arrowprops={"arrowstyle": "->", "color": "black"},
                     fontsize=8, ha="left")

    plt.tight_layout()
    _fig_path = "grasp_poly_grados.png"
    plt.savefig(_fig_path, dpi=150, bbox_inches="tight")
    plt.close()
    from IPython.display import Image as _Img, display as _disp
    _disp(_Img(_fig_path))
    print(f"  Figura guardada en {_fig_path}")


  [comparativa omitida] _MEJOR_GRADO = 2 (por defecto)


### Benchmark GRASP (alfa fijo, base) vs GRASP+ML

Compara **GRASP** con $\alpha=0.5$ fijo (`ejecutar_grasp`, `usar_alfa_adaptativo=False`)
contra **GRASP+ML** con el selector polinómico de grado `_MEJOR_GRADO`
(`ejecutar_grasp_ml`), sobre `wlp01`–`wlp04`, 5 semillas por instancia, 50
iteraciones GRASP para ambos (15 de exploración inicial en la variante ML,
igual que en la comparación de grados polinómicos de más arriba).


In [351]:
import os as _os_gab
import time as _tgab
import io as _io_gab
import contextlib as _ctx_gab
import numpy as np

_EJECUTAR_BENCHMARK_GRASP_ML = False  # cambia a True para ejecutar este benchmark (lento)

if _EJECUTAR_BENCHMARK_GRASP_ML:

    _INST_GRASP_BENCH    = ["wlp01.dzn", "wlp02.dzn", "wlp03.dzn", "wlp04.dzn"]
    _REF_GRASP_BENCH      = {"wlp01": 28716, "wlp02": 52952, "wlp03": 64296, "wlp04": 84633}
    _SEMILLAS_GRASP_BENCH = [0, 1, 2, 3, 4]
    _N_ITER_GRASP_BENCH   = 50
    _MIN_MUESTRAS_GRASP   = 15

    _grasp_res = {
        nombre: {"GRASP": {"gaps": [], "t": []},
                 "GRASP+ML": {"gaps": [], "t": []}}
        for nombre in [a.replace(".dzn", "") for a in _INST_GRASP_BENCH]
    }

    for archivo in _INST_GRASP_BENCH:
        nombre = archivo.replace(".dzn", "")
        datos  = leer_instancia_dzn(_os_gab.path.join("Instaces_MS-CFLP-CI/Public", archivo))
        ref    = _REF_GRASP_BENCH[nombre]

        print(f"\n  {nombre}")
        for semilla in _SEMILLAS_GRASP_BENCH:
            # ── GRASP base (alfa=0.5 fijo) ─────────────────────────────────────
            # ejecutar_grasp no tiene parámetro verbose, así que se silencia su
            # impresión por iteración redirigiendo stdout durante la llamada.
            t0 = _tgab.perf_counter()
            with _ctx_gab.redirect_stdout(_io_gab.StringIO()):
                r_base = ejecutar_grasp(
                    datos["numero_instalaciones"], datos["numero_clientes"],
                    datos["capacidad_instalacion"], datos["costo_fijo_instalacion"],
                    datos["demanda_cliente"], datos["costo_envio"],
                    alfa=0.5, numero_maximo_de_iteraciones=_N_ITER_GRASP_BENCH,
                    semilla_aleatoria=semilla, usar_alfa_adaptativo=False,
                    pares_incompatibles=datos["pares_incompatibles"],
                )
            t_base   = _tgab.perf_counter() - t0
            gap_base = 100 * (r_base["costo_total_minimo"] - ref) / ref
            _grasp_res[nombre]["GRASP"]["gaps"].append(gap_base)
            _grasp_res[nombre]["GRASP"]["t"].append(t_base)

            # ── GRASP+ML (selector polinómico, grado _MEJOR_GRADO) ─────────────
            r_ml = ejecutar_grasp_ml(
                datos, grado=_MEJOR_GRADO, n_iter=_N_ITER_GRASP_BENCH,
                min_muestras=_MIN_MUESTRAS_GRASP, semilla=semilla, verbose=False,
            )
            gap_ml = 100 * (r_ml.costo_final - ref) / ref
            _grasp_res[nombre]["GRASP+ML"]["gaps"].append(gap_ml)
            _grasp_res[nombre]["GRASP+ML"]["t"].append(r_ml.tiempo_total_s)

            print(f"    seed={semilla}  GRASP: {gap_base:+.2f}% {t_base:.1f}s  "
                  f"GRASP+ML: {gap_ml:+.2f}% {r_ml.tiempo_total_s:.1f}s  "
                  f"Δgap={gap_ml-gap_base:+.2f}pp")

    # ── Tabla resumen ─────────────────────────────────────────────────────────────
    instancias_grasp = [a.replace(".dzn", "") for a in _INST_GRASP_BENCH]
    print(f"\n{'='*70}")
    print(f"  {'':<10}  {'── GRASP base ──':^18}  {'── GRASP+ML ──':^18}  {'Δgap':>6}")
    print(f"  {'Inst.':<10}  {'gap%':>7}  {'σ':>5}  {'t(s)':>5}  "
          f"{'gap%':>7}  {'σ':>5}  {'t(s)':>5}  {'(pp)':>6}")
    print(f"  {'-'*70}")

    all_base, all_ml = [], []
    for nombre in instancias_grasp:
        bg = np.mean(_grasp_res[nombre]["GRASP"]["gaps"])
        bs = np.std(_grasp_res[nombre]["GRASP"]["gaps"])
        bt = np.mean(_grasp_res[nombre]["GRASP"]["t"])
        mg = np.mean(_grasp_res[nombre]["GRASP+ML"]["gaps"])
        ms = np.std(_grasp_res[nombre]["GRASP+ML"]["gaps"])
        mt = np.mean(_grasp_res[nombre]["GRASP+ML"]["t"])
        all_base.append(bg); all_ml.append(mg)
        print(f"  {nombre:<10}  {bg:>+7.2f}  {bs:>5.2f}  {bt:>5.1f}  "
              f"{mg:>+7.2f}  {ms:>5.2f}  {mt:>5.1f}  {mg-bg:>+6.2f}")

    print(f"  {'-'*70}")
    mb = np.mean(all_base); mm = np.mean(all_ml)
    print(f"  {'Media':<10}  {mb:>+7.2f}  {'':<5}  {'':<5}  {mm:>+7.2f}  {'':<5}  {'':<5}  {mm-mb:>+6.2f}")
    print(f"  {'='*70}")
else:
    print("Benchmark GRASP vs GRASP+ML omitido (_EJECUTAR_BENCHMARK_GRASP_ML = False)")


Benchmark GRASP vs GRASP+ML omitido (_EJECUTAR_BENCHMARK_GRASP_ML = False)


## 3. GVNS (General Variable Neighborhood Search)

Alterna perturbación (shaking con radio creciente k) y búsqueda local (VND)
hasta converger. Se usa como refinador sobre la solución de GRASP o la seleccionada por el RF.

```
k <- 1
mientras k <= k_max:
    s' <- Shaking(s, k)
    s'' <- VND(s')
    si coste(s'') < coste(s): s <- s'', k <- 1
    si no:                     k <- k + 1
```

### Estructuras de datos (GVNS)

In [352]:
class ResultadoGVNS(TypedDict):
    """Resultado devuelto por el algoritmo GVNS."""

    costo_total_minimo: float
    instalaciones_abiertas: list[bool]
    cliente_asignado_a_instalacion: list[int]
    capacidad_restante_por_instalacion: list[int]
    iteraciones_con_mejora: int

### Componentes de GVNS

Cinco funciones en secuencia: `sacudir_solucion` (perturbación);
`_mejorar_con_1_shift` (N1, primera mejora, específico de GVNS);
`_cerrar_instalaciones_poco_rentables` (N2) y `_abrir_instalaciones_rentables`
(N3) —definidas en la sección~1, Búsqueda local, y reutilizadas aquí tal
cual—; coordinadas por `descenso_por_vecindarios_variables` (VND).

#### Shaking

Reasigna k clientes al azar a instalaciones factibles alternativas.
El GVNS incrementa k cuando VND no mejora y lo reinicia a 1 cuando sí.

In [353]:
def sacudir_solucion(
    instalacion_esta_abierta: list[bool],
    instalacion_asignada_al_cliente: list[int],
    capacidad_restante_instalacion: list[int],
    numero_instalaciones: int,
    numero_clientes: int,
    demanda_cliente: list[int],
    k: int,
    incompatibles_cliente: list[set[int]] | None = None,
) -> SolucionCFLP:
    """Shaking: reasigna k clientes al azar a instalaciones factibles alternativas."""
    instalacion_esta_abierta = list(instalacion_esta_abierta)
    instalacion_asignada_al_cliente = list(instalacion_asignada_al_cliente)
    capacidad_restante_instalacion = list(capacidad_restante_instalacion)

    clientes_a_perturbar = random.sample(
        range(numero_clientes), min(k, numero_clientes)
    )

    for indice_cliente in clientes_a_perturbar:
        demanda = demanda_cliente[indice_cliente]
        instalacion_origen = instalacion_asignada_al_cliente[indice_cliente]
        inc = incompatibles_cliente[indice_cliente] if incompatibles_cliente else set()

        candidatos_destino = [
            i for i in range(numero_instalaciones)
            if i != instalacion_origen
            and capacidad_restante_instalacion[i] >= demanda
            and not any(instalacion_asignada_al_cliente[k] == i for k in inc)
        ]

        if not candidatos_destino:
            continue

        instalacion_destino = random.choice(candidatos_destino)

        capacidad_restante_instalacion[instalacion_origen] += demanda
        capacidad_restante_instalacion[instalacion_destino] -= demanda
        instalacion_asignada_al_cliente[indice_cliente] = instalacion_destino
        instalacion_esta_abierta[instalacion_destino] = True

        sigue_ocupada = any(
            instalacion_asignada_al_cliente[j] == instalacion_origen
            for j in range(numero_clientes)
        )
        if not sigue_ocupada:
            instalacion_esta_abierta[instalacion_origen] = False

    return (
        instalacion_esta_abierta,
        instalacion_asignada_al_cliente,
        capacidad_restante_instalacion,
    )

#### N1 en GVNS: primera mejora

GVNS necesita una variante de N1 distinta a la de GRASP: en vez de barrer
todos los clientes reasignando cada mejora que encuentra
(`_reasignar_clientes_en_barrido`, sección~1), se detiene en el primer
1-shift que reduzca el coste y devuelve el control de inmediato. Esta
política de "primera mejora, un único movimiento" es la que necesita el VND
para poder reiniciar desde N1 después de cada paso y escalar a N2/N3 solo
cuando N1 ya no encuentra nada — con el barrido completo de GRASP, cada
llamada a N1 haría de por sí varias reasignaciones antes de devolver el
control, cambiando cómo VND intercala los tres vecindarios. El delta que usa
es el mismo `_delta_reasignar_cliente` de la sección~1; solo cambia cuándo
parar.

In [354]:
def _mejorar_con_1_shift(
    instalacion_esta_abierta: list[bool],
    instalacion_asignada_al_cliente: list[int],
    capacidad_restante_instalacion: list[int],
    numero_instalaciones: int,
    numero_clientes: int,
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    incompatibles_cliente: list[set[int]] | None = None,
) -> tuple[SolucionCFLP, bool]:
    """N1 first-improvement, específico de GVNS: se detiene en el primer
    1-shift que reduzca el coste y devuelve el control de inmediato (a
    diferencia de `_reasignar_clientes_en_barrido`, que sigue barriendo).
    Usa el mismo delta de N1 (sección 1, Búsqueda local); solo cambia la
    política de cuándo parar."""
    instalacion_esta_abierta = list(instalacion_esta_abierta)
    instalacion_asignada_al_cliente = list(instalacion_asignada_al_cliente)
    capacidad_restante_instalacion = list(capacidad_restante_instalacion)

    # Ocupación de cada instalación, precalculada una sola vez: como la función
    # devuelve el control en el primer movimiento aceptado, la asignación no
    # cambia durante el barrido y esta cuenta se mantiene válida para todos los
    # clientes evaluados (evita recontar en O(n) para cada uno de los n clientes).
    carga_por_instalacion = [0] * numero_instalaciones
    for j in range(numero_clientes):
        carga_por_instalacion[instalacion_asignada_al_cliente[j]] += 1

    for indice_cliente in range(numero_clientes):
        instalacion_actual = instalacion_asignada_al_cliente[indice_cliente]
        demanda = demanda_cliente[indice_cliente]
        inc = incompatibles_cliente[indice_cliente] if incompatibles_cliente else set()
        # Instalaciones vetadas para este cliente: se calcula una vez por cliente
        # (O(|inc|)) en lugar de recorrer `inc` para cada una de las m
        # instalaciones candidatas (O(m * |inc|)).
        instalaciones_prohibidas = {instalacion_asignada_al_cliente[k] for k in inc}

        es_el_unico_en_origen = carga_por_instalacion[instalacion_actual] == 1

        for instalacion_destino in range(numero_instalaciones):
            if instalacion_destino == instalacion_actual:
                continue
            if capacidad_restante_instalacion[instalacion_destino] < demanda:
                continue
            if instalacion_destino in instalaciones_prohibidas:
                continue

            variacion_de_costo = _delta_reasignar_cliente(
                indice_cliente, instalacion_actual, instalacion_destino,
                instalacion_esta_abierta, costo_fijo_instalacion, demanda_cliente,
                costo_envio, es_el_unico_en_origen,
            )

            if variacion_de_costo < 0:
                capacidad_restante_instalacion[instalacion_actual] += demanda
                capacidad_restante_instalacion[instalacion_destino] -= demanda
                instalacion_asignada_al_cliente[indice_cliente] = (
                    instalacion_destino
                )
                instalacion_esta_abierta[instalacion_destino] = True
                if es_el_unico_en_origen:
                    instalacion_esta_abierta[instalacion_actual] = False
                return (
                    instalacion_esta_abierta,
                    instalacion_asignada_al_cliente,
                    capacidad_restante_instalacion,
                ), True

    return (
        instalacion_esta_abierta,
        instalacion_asignada_al_cliente,
        capacidad_restante_instalacion,
    ), False


#### VND: descenso por vecindarios variables

Prueba N1 (O(n·m)) antes de N2 y N3 (cierre/apertura de instalaciones, mismo orden de
complejidad que N1 pero con una constante mayor) y reinicia desde N1 tras cada mejora.
Para cuando ninguno de los tres produce mejora.

In [355]:
def descenso_por_vecindarios_variables(
    instalacion_esta_abierta: list[bool],
    instalacion_asignada_al_cliente: list[int],
    capacidad_restante_instalacion: list[int],
    numero_instalaciones: int,
    numero_clientes: int,
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    incompatibles_cliente: list[set[int]] | None = None,
) -> SolucionCFLP:
    """VND: alterna N1 (1-shift), N2 (cierre) y N3 (apertura) reiniciando desde N1 tras cada mejora, hasta mínimo local."""
    solucion: SolucionCFLP = (
        list(instalacion_esta_abierta),
        list(instalacion_asignada_al_cliente),
        list(capacidad_restante_instalacion),
    )

    while True:
        solucion, mejoro_n1 = _mejorar_con_1_shift(
            solucion[0], solucion[1], solucion[2],
            numero_instalaciones, numero_clientes,
            costo_fijo_instalacion, demanda_cliente, costo_envio,
            incompatibles_cliente,
        )
        if mejoro_n1:
            continue

        solucion, mejoro_n2 = _cerrar_instalaciones_poco_rentables(
            solucion[0], solucion[1], solucion[2],
            numero_instalaciones, numero_clientes,
            costo_fijo_instalacion, demanda_cliente, costo_envio,
            incompatibles_cliente,
        )
        if mejoro_n2:
            continue

        solucion, mejoro_n3 = _abrir_instalaciones_rentables(
            solucion[0], solucion[1], solucion[2],
            numero_instalaciones, numero_clientes,
            costo_fijo_instalacion, demanda_cliente, costo_envio,
            incompatibles_cliente,
        )
        if mejoro_n3:
            continue

        break

    return solucion

### Ejecución de GVNS

In [356]:
def ejecutar_gvns(
    instalacion_esta_abierta: list[bool],
    instalacion_asignada_al_cliente: list[int],
    capacidad_restante_instalacion: list[int],
    numero_instalaciones: int,
    numero_clientes: int,
    capacidad_instalacion: list[int],
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    k_maximo: int = 3,
    numero_maximo_de_iteraciones: int = 30,
    semilla_aleatoria: int = 0,
    pares_incompatibles: list[tuple[int, int]] | None = None,
) -> ResultadoGVNS:
    """GVNS: shaking con k creciente + VND hasta convergencia."""
    random.seed(semilla_aleatoria)

    incompatibles = _precomputar_incompatibles(numero_clientes, pares_incompatibles or [])

    mejor_solucion: SolucionCFLP = (
        list(instalacion_esta_abierta),
        list(instalacion_asignada_al_cliente),
        list(capacidad_restante_instalacion),
    )
    mejor_costo = calcular_costo_total(
        mejor_solucion[0],
        mejor_solucion[1],
        costo_fijo_instalacion,
        demanda_cliente,
        costo_envio,
    )
    iteraciones_con_mejora = 0

    print("=" * 65)
    print("  GVNS - General Variable Neighborhood Search (MS-CFLP-CI)")
    print("=" * 65)
    print(f"  Instalaciones      : {numero_instalaciones}")
    print(f"  Clientes           : {numero_clientes}")
    print(f"  Pares incompatibles: {len(pares_incompatibles or [])}")
    print(f"  k máximo (shaking) : {k_maximo}")
    print(f"  Iteraciones máx.   : {numero_maximo_de_iteraciones}")
    print(f"  Costo inicial      : {mejor_costo:,.0f}")
    print("=" * 65)

    for iteracion in range(1, numero_maximo_de_iteraciones + 1):
        k = 1
        mejoro_en_esta_iteracion = False

        while k <= k_maximo:
            solucion_agitada = sacudir_solucion(
                mejor_solucion[0],
                mejor_solucion[1],
                mejor_solucion[2],
                numero_instalaciones,
                numero_clientes,
                demanda_cliente,
                k,
                incompatibles,
            )

            solucion_refinada = descenso_por_vecindarios_variables(
                solucion_agitada[0],
                solucion_agitada[1],
                solucion_agitada[2],
                numero_instalaciones,
                numero_clientes,
                costo_fijo_instalacion,
                demanda_cliente,
                costo_envio,
                incompatibles,
            )

            costo_refinado = calcular_costo_total(
                solucion_refinada[0],
                solucion_refinada[1],
                costo_fijo_instalacion,
                demanda_cliente,
                costo_envio,
            )

            if costo_refinado < mejor_costo:
                mejor_costo = costo_refinado
                mejor_solucion = (
                    list(solucion_refinada[0]),
                    list(solucion_refinada[1]),
                    list(solucion_refinada[2]),
                )
                k = 1
                mejoro_en_esta_iteracion = True
                iteraciones_con_mejora += 1
                print(
                    f"  Iter {iteracion:>3} | k={k} | "
                    f"Mejor: {mejor_costo:>10,.0f} << NUEVA MEJOR"
                )
            else:
                k += 1

        if not mejoro_en_esta_iteracion and (iteracion % 5 == 0):
            print(
                f"  Iter {iteracion:>3} | Sin mejora | "
                f"Mejor: {mejor_costo:>10,.0f}"
            )

    print("=" * 65)
    print(f"  SOLUCIÓN FINAL - Costo total: {mejor_costo:,.0f}")
    print(f"  Iteraciones con mejora: {iteraciones_con_mejora}")
    print("=" * 65)

    return {
        "costo_total_minimo": mejor_costo,
        "instalaciones_abiertas": list(mejor_solucion[0]),
        "cliente_asignado_a_instalacion": list(mejor_solucion[1]),
        "capacidad_restante_por_instalacion": list(mejor_solucion[2]),
        "iteraciones_con_mejora": iteraciones_con_mejora,
    }

### GVNS + Q-Learning: selección adaptativa de vecindarios

En el VND clásico el orden de aplicación de vecindarios es **fijo**: primero se
agota N1 (1-shift), luego N2 (cierre de instalaciones) y por último N3 (apertura
de instalaciones); el descenso reinicia desde N1 tras cualquier mejora y se
declara mínimo local cuando ninguno de los tres mejora. Este orden es razonable
en promedio, pero *paga siempre* el coste de invocar N2 y N3 —ambos recorren
todas las instalaciones abiertas o cerradas— incluso en las muchas iteraciones
donde N1 por sí solo ya alcanza el óptimo local.

**GVNS+QL** sustituye ese orden fijo por una política aprendida mediante
**Q-learning tabular**. En cada bloque del descenso, un agente decide qué
vecindario *agotar* según el estado actual (qué vecindario se agotó en el bloque
anterior y si mejoró). La política es ε-greedy con ε decreciente: explora los
tres vecindarios al principio y converge hacia la secuencia que maximiza la
mejora **por unidad de tiempo**.

> **Nota.** Los resultados empíricos de la comparación GVNS+VND vs. GVNS+QL
> (sección más abajo) están pendientes de reejecutar con el portafolio de tres
> vecindarios: las cifras de una versión anterior de este cuaderno (con
> portafolio de solo dos acciones) ya no aplican tras incorporar N3.


### Formulación del agente Q-learning

**Vecindarios del portafolio** — 0 = N1 (1-shift, reasignación individual;
barato), 1 = N2 (cierre de instalaciones poco rentables), 2 = N3 (apertura de
instalaciones rentables). Se descarta el intercambio de pares (2-swap,
$O(n^2)$) del portafolio: N1, N2 y N3 comparten el mismo orden de complejidad
$O(n \cdot m)$ pero con efectos estructurales distintos (reasignar, consolidar,
expandir), lo que da al agente tres decisiones con contenido real.

**Acción = agotar un vecindario (bloque), no un movimiento suelto.** Cada acción
aplica el operador *first-improvement* (o, en el caso de N2/N3, el barrido
completo) elegido en bucle hasta que deja de mejorar. Esto tiene dos
consecuencias frente a la formulación por-movimiento:

- La **recompensa** es la mejora de un *bloque completo* de descenso — una señal
  grande y estable — en lugar de la mejora minúscula de un solo movimiento (que
  quedaba dominada por la penalización de fallo).
- El descenso termina, como el VND, en un **mínimo local garantizado** (los tres
  vecindarios agotados), sin depender de una cota artificial de pasos.

**Estados** — capturan la dinámica reciente del descenso:

| Estado | Descripción |
|--------|-------------|
| arranque | primer bloque del descenso (sin historial) |
| (N1, ✗) | último bloque fue N1 y no mejoró |
| (N1, ✓) | último bloque fue N1 y mejoró |
| (N2, ✗) | último bloque fue N2 y no mejoró |
| (N2, ✓) | último bloque fue N2 y mejoró |
| (N3, ✗) | último bloque fue N3 y no mejoró |
| (N3, ✓) | último bloque fue N3 y mejoró |

**Enmascaramiento de acciones** — un vecindario recién agotado sin mejora queda
*excluido* hasta que alguno de los vecindarios no excluidos logre una mejora,
momento en el que la exclusión se reinicia y los demás vuelven a estar
disponibles. Como los operadores son deterministas, reintentar uno que acaba de
fallar sobre la misma solución fallaría con certeza; el enmascaramiento elimina
ese desperdicio y hace que el estado "los tres excluidos" signifique
inequívocamente *mínimo local*.

**Recompensa** — mejora relativa del bloque, **normalizada por tiempo** para que
el agente prefiera el vecindario más eficiente cuando varios mejoran de forma
comparable ($t_{\text{ref}}$ es una constante de referencia que evita divisiones
por tiempos casi nulos):

$$r_t = \begin{cases}
  \dfrac{C_{t-1} - C_t}{C_{t-1}} \cdot \dfrac{t_{\text{ref}}}{\Delta t + t_{\text{ref}}} & \text{si el bloque mejoró} \\[10pt]
  -0.001 \cdot \dfrac{\Delta t + t_{\text{ref}}}{t_{\text{ref}}} & \text{en caso contrario}
\end{cases}$$

La penalización por fallo se reduce a $-0.001$ (frente a $-0.01$) para que **no
domine** la señal de las mejoras: con movimientos individuales la mejora mediana
medida era $\approx 0.002$, de modo que una penalización de $-0.01$ la superaba
en $5\times$ y empujaba al agente a *evitar* descender. Al recompensar bloques
completos, la mejora típica es mucho mayor y el equilibrio se restablece.

**Actualización Q-learning** (regla TD off-policy):

$$Q(s,a) \;\leftarrow\; Q(s,a) + \alpha\!\left[\,r + \gamma\max_{a'}Q(s',a') - Q(s,a)\right]$$

**Parámetros**: $\alpha = 0.1$, $\gamma = 0.8$, $\varepsilon_0 = 0.3$,
$\varepsilon_{\min} = 0.05$, $\varepsilon_{\text{decay}} = 0.99$. El agente usa un
**RNG propio** para la política ε-greedy, de modo que activar QL no altera la
secuencia de shaking (que depende del RNG global) y la comparación base-vs-QL es
justa (misma trayectoria de perturbaciones por semilla).

In [357]:
import numpy as np

class SelectorQLearning:
    """Selecciona adaptativamente qué vecindario AGOTAR (N1, N2 ó N3) mediante
    Q-learning tabular con enmascaramiento de acciones.

    Portafolio: acción 0 = N1 (1-shift), acción 1 = N2 (cierre de instalaciones),
    acción 2 = N3 (apertura de instalaciones).
    Estado: (último vecindario agotado {None=arranque, 0=N1, 1=N2, 2=N3},
             ¿mejoró? {False, True})  ->  7 estados.
    Tabla Q: 7 estados x 3 acciones = 21 entradas.

    Correcciones frente a la versión por-movimiento:
      1. La acción agota un vecindario completo (bloque), no un solo movimiento.
      2. `seleccionar_accion` admite acciones excluidas (enmascaramiento): nunca
         se reintenta un vecindario que acaba de fallar sobre la misma solución.
      3. Penalización de fallo reducida a -0.001 (no domina la recompensa).
      4. Sin sesgo inicial en Q y RNG propio (no interfiere con el shaking).

    `t_ref` (duración "típica" de bloque, usada para normalizar la recompensa por
    tiempo) ya no es una constante fija: se mantiene como media móvil exponencial
    de la duración real de los bloques observados en esta misma ejecución, por lo
    que se autocalibra a la escala de tiempo de cada instancia en lugar de asumir
    un valor global (ver `actualizar_t_ref`).
    """

    N_VEC = 3  # N1, N2 y N3
    NOMBRES_VECINDARIOS = ("N1", "N2", "N3")
    TASA_ACTUALIZACION_T_REF = 0.1  # peso de la media móvil exponencial

    def __init__(self, alpha: float = 0.2, gamma: float = 0.8,
                 epsilon: float = 0.5, epsilon_min: float = 0.05,
                 epsilon_decay: float = 0.999, semilla: int = 0) -> None:
        self.alpha         = alpha
        self.gamma         = gamma
        self.epsilon       = epsilon
        self.epsilon_min   = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.Q = np.zeros((self.N_VEC * 2 + 1, self.N_VEC))  # sin sesgo inicial
        self.rng = random.Random(semilla)  # RNG propio: no toca la secuencia de shaking
        self.n_pasos = 0
        self.t_ref: float | None = None  # sin calibrar hasta el primer bloque

    def codificar_estado(self, ultimo_vec: int | None, mejoro: bool) -> int:
        if ultimo_vec is None:
            return self.N_VEC * 2   # estado de arranque
        return ultimo_vec * 2 + (1 if mejoro else 0)

    def seleccionar_accion(self, estado: int, excluidas: frozenset = frozenset()) -> int:
        """Política ε-greedy con enmascaramiento de acciones agotadas."""
        candidatas = [a for a in range(self.N_VEC) if a not in excluidas]
        if len(candidatas) == 1:
            return candidatas[0]
        if self.rng.random() < self.epsilon:
            return self.rng.choice(candidatas)
        return max(candidatas, key=lambda a: self.Q[estado, a])

    def actualizar_t_ref(self, duracion_bloque: float) -> None:
        """Actualiza la media móvil exponencial de duración de bloque.

        El primer bloque observado calibra `t_ref` directamente con su propia
        duración (no hay historial previo); los siguientes lo desplazan con
        peso `TASA_ACTUALIZACION_T_REF` hacia la duración recién medida, de
        modo que `t_ref` converge hacia la duración típica de bloque en esta
        instancia y esta ejecución, en lugar de asumir un valor fijo.
        """
        if self.t_ref is None:
            self.t_ref = duracion_bloque
        else:
            self.t_ref = (
                (1 - self.TASA_ACTUALIZACION_T_REF) * self.t_ref
                + self.TASA_ACTUALIZACION_T_REF * duracion_bloque
            )

    def actualizar(self, estado: int, accion: int,
                   recompensa: float, estado_sig: int) -> None:
        """Regla de actualización Q-learning (TD off-policy)."""
        mejor_futuro = float(np.max(self.Q[estado_sig]))
        td_error = recompensa + self.gamma * mejor_futuro - self.Q[estado, accion]
        self.Q[estado, accion] += self.alpha * td_error
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        self.n_pasos += 1

    def resumen(self) -> str:
        q = self.Q
        nombres = self.NOMBRES_VECINDARIOS

        def fila(indice_estado: int, etiqueta: str) -> str:
            valores = ",".join(f"{q[indice_estado, a]:.3f}" for a in range(self.N_VEC))
            mejor = nombres[int(np.argmax(q[indice_estado]))]
            return f"Q({etiqueta})=[{valores}]->{mejor}"

        arranque = self.N_VEC * 2
        t_ref_str = f"{self.t_ref*1000:.1f}ms" if self.t_ref is not None else "sin calibrar"
        partes = [f"ε={self.epsilon:.3f} | pasos={self.n_pasos} | t_ref={t_ref_str}",
                  fila(arranque, "arr")]
        for indice_vecindario, nombre in enumerate(nombres):
            partes.append(fila(indice_vecindario * 2, f"{nombre}✗"))
            partes.append(fila(indice_vecindario * 2 + 1, f"{nombre}✓"))
        return " | ".join(partes)

In [358]:
import time as _t_ql

def seleccionar_solucion_aleatoria_del_pool(
    datos: DatosCFLP,
    tamano_pool: int = 5,
    semilla_aleatoria: int = 0,
) -> SolucionCFLP:
    """Genera un pool de soluciones GRASP y devuelve una elegida al azar."""
    random.seed(semilla_aleatoria)
    incompatibles = _precomputar_incompatibles(
        datos["numero_clientes"], datos["pares_incompatibles"]
    )
    pool = [
        construir_solucion_greedy_aleatoria(
            datos["numero_instalaciones"],
            datos["numero_clientes"],
            datos["capacidad_instalacion"],
            datos["costo_fijo_instalacion"],
            datos["demanda_cliente"],
            datos["costo_envio"],
            round(random.uniform(0.0, 1.0), 2),
            incompatibles,
        )
        for _ in range(tamano_pool)
    ]
    return random.choice(pool)


def _ejecutar_gvns_iter(
    datos: DatosCFLP,
    abierta: list[bool],
    asignada: list[int],
    restante: list[int],
    k_maximo: int,
    n_iter: int,
    semilla: int,
    max_mejoras: int | None = None,
) -> float:
    """GVNS con número fijo de iteraciones externas.

    Usa mejorar_solucion_con_busqueda_local (reasignación en barrido + cierre)
    en lugar del VND de primera mejora, para tiempos razonables en instancias
    grandes. max_mejoras limita el total de mejoras aceptadas (por defecto sin
    límite efectivo).
    """
    if max_mejoras is None:
        max_mejoras = n_iter * k_maximo  # cota holgada: nunca restrictiva
    random.seed(semilla)
    F    = datos["numero_instalaciones"]
    C    = datos["numero_clientes"]
    cap  = datos["capacidad_instalacion"]
    cf   = datos["costo_fijo_instalacion"]
    dem  = datos["demanda_cliente"]
    envio = datos["costo_envio"]
    incompatibles = _precomputar_incompatibles(C, datos["pares_incompatibles"])

    mejor_sol = (list(abierta), list(asignada), list(restante))
    mejor_costo = calcular_costo_total(mejor_sol[0], mejor_sol[1], cf, dem, envio)
    mejoras_total = 0

    for _ in range(n_iter):
        k = 1
        while k <= k_maximo and mejoras_total < max_mejoras:
            agitada = sacudir_solucion(
                mejor_sol[0], mejor_sol[1], mejor_sol[2], F, C, dem, k, incompatibles
            )
            refinada = mejorar_solucion_con_busqueda_local(
                agitada[0], agitada[1], agitada[2], F, C, cap, cf, dem, envio, incompatibles
            )
            costo_ref = calcular_costo_total(refinada[0], refinada[1], cf, dem, envio)
            if costo_ref < mejor_costo:
                mejor_costo   = costo_ref
                mejor_sol     = (list(refinada[0]), list(refinada[1]), list(refinada[2]))
                mejoras_total += 1
                k = 1
            else:
                k += 1

    return mejor_costo







def _ejecutar_gvns_iter_vnd(
    datos: DatosCFLP,
    abierta: list[bool],
    asignada: list[int],
    restante: list[int],
    k_maximo: int,
    n_iter: int,
    semilla: int,
    max_mejoras: int | None = None,
    tiempo_limite_s: float | None = None,
) -> float:
    """GVNS con VND completo (N1→N2→N3 orden fijo). Base de comparación para GVNS+QL,
    que solo elige entre N1 y N2 (ver nota en la sección de GVNS+QL).

    Si se indica `tiempo_limite_s`, la búsqueda también se detiene al agotar
    ese tiempo, lo que ocurra primero."""
    if max_mejoras is None:
        max_mejoras = n_iter * k_maximo
    random.seed(semilla)
    t_inicio = time.perf_counter()
    F, C = datos["numero_instalaciones"], datos["numero_clientes"]
    cf   = datos["costo_fijo_instalacion"]
    dem  = datos["demanda_cliente"]
    envio = datos["costo_envio"]
    incompatibles = _precomputar_incompatibles(C, datos["pares_incompatibles"])

    mejor_sol   = (list(abierta), list(asignada), list(restante))
    mejor_costo = calcular_costo_total(mejor_sol[0], mejor_sol[1], cf, dem, envio)
    mejoras_total = 0

    for _ in range(n_iter):
        if mejoras_total >= max_mejoras:
            break
        if tiempo_limite_s is not None and time.perf_counter() - t_inicio >= tiempo_limite_s:
            break
        ya_probado: set[int] = set()
        while len(ya_probado) < k_maximo:
            if tiempo_limite_s is not None and time.perf_counter() - t_inicio >= tiempo_limite_s:
                break
            k = 1 if 1 not in ya_probado else min(
                j for j in range(1, k_maximo + 1) if j not in ya_probado
            )
            agitada  = sacudir_solucion(
                mejor_sol[0], mejor_sol[1], mejor_sol[2], F, C, dem, k, incompatibles
            )
            refinada = descenso_por_vecindarios_variables(
                agitada[0], agitada[1], agitada[2], F, C, cf, dem, envio, incompatibles
            )
            costo_ref = calcular_costo_total(refinada[0], refinada[1], cf, dem, envio)
            if costo_ref < mejor_costo:
                mejor_costo   = costo_ref
                mejor_sol     = refinada
                mejoras_total += 1
                ya_probado.clear()
                if mejoras_total >= max_mejoras:
                    break
            else:
                ya_probado.add(k)
    return mejor_costo



def _agotar_vecindario(operador, sol, F, C, cf, dem, envio, incompatibles):
    """Aplica un operador first-improvement en bucle hasta que deja de mejorar.

    Devuelve (solucion, mejoro_alguna_vez). Convierte un operador de *un
    movimiento* en un *bloque* de descenso completo sobre ese vecindario.
    """
    mejoro_alguna = False
    while True:
        sol, mejoro = operador(sol[0], sol[1], sol[2],
                               F, C, cf, dem, envio, incompatibles)
        if not mejoro:
            return sol, mejoro_alguna
        mejoro_alguna = True


def _ejecutar_gvns_iter_ql(
    datos: DatosCFLP,
    abierta: list[bool],
    asignada: list[int],
    restante: list[int],
    k_maximo: int,
    n_iter: int,
    semilla: int,
    max_mejoras: int | None = None,
    tiempo_limite_s: float | None = None,
) -> tuple[float, SelectorQLearning]:
    """GVNS donde el descenso usa Q-learning para decidir qué vecindario AGOTAR.

    Portafolio {N1 = 1-shift, N2 = cierre de instalaciones, N3 = apertura de
    instalaciones}. Sustituye el VND de orden fijo N1->N2->N3 por una política
    aprendida. El descenso termina en mínimo local (los tres vecindarios
    agotados), igual que el VND, pero el agente aprende a evitar las pasadas
    caras de N2/N3 cuando N1 ya basta.

    La duración de referencia usada para normalizar la recompensa por tiempo
    (`t_ref`) ya no es una constante fija: el agente la calibra online como
    media móvil de la duración real de los bloques (`agente.actualizar_t_ref`),
    autoajustándose a la escala de tiempo de cada instancia.

    Devuelve (mejor_costo, agente) — el agente permite inspeccionar la política
    aprendida vía `agente.resumen()`.
    """
    if max_mejoras is None:
        max_mejoras = n_iter * k_maximo
    random.seed(semilla)
    t_inicio = time.perf_counter()
    F, C = datos["numero_instalaciones"], datos["numero_clientes"]
    cf    = datos["costo_fijo_instalacion"]
    dem   = datos["demanda_cliente"]
    envio = datos["costo_envio"]
    incompatibles = _precomputar_incompatibles(C, datos["pares_incompatibles"])
    operadores = [_mejorar_con_1_shift, _cerrar_instalaciones_poco_rentables, _abrir_instalaciones_rentables]

    agente      = SelectorQLearning(semilla=semilla + 7919)
    mejor_sol   = (list(abierta), list(asignada), list(restante))
    mejor_costo = calcular_costo_total(mejor_sol[0], mejor_sol[1], cf, dem, envio)
    mejoras_total = 0

    for _ in range(n_iter):
        if mejoras_total >= max_mejoras:
            break
        if tiempo_limite_s is not None and time.perf_counter() - t_inicio >= tiempo_limite_s:
            break
        ya_probado: set[int] = set()

        while len(ya_probado) < k_maximo:
            if tiempo_limite_s is not None and time.perf_counter() - t_inicio >= tiempo_limite_s:
                break
            k = 1 if 1 not in ya_probado else min(
                j for j in range(1, k_maximo + 1) if j not in ya_probado
            )
            sol_actual   = sacudir_solucion(
                mejor_sol[0], mejor_sol[1], mejor_sol[2], F, C, dem, k, incompatibles
            )
            costo_actual = calcular_costo_total(sol_actual[0], sol_actual[1], cf, dem, envio)

            # ── Descenso: VND con orden aprendido por Q-learning ──────────────
            ultimo_vec: int | None = None
            mejoro_bloque          = False
            agotados: set[int]     = set()   # vecindarios sin mejora sobre sol_actual

            while len(agotados) < agente.N_VEC:
                estado = agente.codificar_estado(ultimo_vec, mejoro_bloque)
                accion = agente.seleccionar_accion(estado, excluidas=frozenset(agotados))

                t0 = _t_ql.perf_counter()
                sol_nueva, mejoro_bloque = _agotar_vecindario(
                    operadores[accion], sol_actual, F, C, cf, dem, envio, incompatibles
                )
                dt = _t_ql.perf_counter() - t0
                # t_ref adaptativo: antes de la primera calibración, el propio
                # bloque actual sirve de referencia (factor neutro); a partir de
                # ahí se usa la media móvil acumulada hasta el bloque anterior.
                t_ref_actual = agente.t_ref if agente.t_ref is not None else dt

                if mejoro_bloque:
                    nuevo_costo = calcular_costo_total(sol_nueva[0], sol_nueva[1], cf, dem, envio)
                    # mejora relativa normalizada por tiempo (eficiencia del bloque)
                    recompensa  = ((costo_actual - nuevo_costo) / costo_actual) * t_ref_actual / (dt + t_ref_actual)
                    sol_actual, costo_actual = sol_nueva, nuevo_costo
                    agotados = {accion}   # este vecindario se agotó; el otro puede reactivarse
                else:
                    recompensa = -0.001 * (dt + t_ref_actual) / t_ref_actual
                    agotados.add(accion)

                agente.actualizar_t_ref(dt)
                estado_sig = agente.codificar_estado(accion, mejoro_bloque)
                agente.actualizar(estado, accion, recompensa, estado_sig)
                ultimo_vec = accion
            # ──────────────────────────────────────────────────────────────────

            if costo_actual < mejor_costo:
                mejor_costo   = costo_actual
                mejor_sol     = sol_actual
                mejoras_total += 1
                ya_probado.clear()
                if mejoras_total >= max_mejoras:
                    break
            else:
                ya_probado.add(k)

    return mejor_costo, agente

### Benchmark GVNS+VND (base) vs GVNS+QL

Compara **GVNS+VND** de orden fijo (N1→N2→N3) contra **GVNS+QL** (mismo
portafolio de tres vecindarios, orden aprendido) sobre la misma solución
inicial y —gracias al RNG propio del agente— la misma secuencia de shaking por
semilla. Se reportan tres magnitudes: *gap* de calidad respecto a la
referencia, tiempo, y el **speedup** de QL sobre la base, que es donde está la
aportación real de la componente ML.

In [359]:
import os as _os_gb
import time as _tgb
import numpy as np

_EJECUTAR_BENCHMARK_GVNS_QL = False  # cambia a True para ejecutar este benchmark (lento)

if _EJECUTAR_BENCHMARK_GVNS_QL:

    _INST_BENCH     = ["wlp01.dzn", "wlp02.dzn", "wlp03.dzn", "wlp04.dzn",
                        "wlp05.dzn", "wlp06.dzn", "wlp07.dzn", "wlp08.dzn"]
    _REF_BENCH      = {"wlp01": 28716, "wlp02": 52952, "wlp03": 64296, "wlp04": 84633,
                        "wlp05": 103857, "wlp06": 111654, "wlp07": 162277, "wlp08": 187938}
    _SEMILLAS_BENCH = [0, 1, 2, 3, 4]
    _K_MAX_BENCH    = 10
    _N_ITER_BENCH   = 30
    _POOL_BENCH     = 10

    _gb_res = {
        nombre: {"GVNS": {"gaps": [], "t": []},
                 "GVNS+QL": {"gaps": [], "t": []}}
        for nombre in [a.replace(".dzn", "") for a in _INST_BENCH]
    }

    for archivo in _INST_BENCH:
        nombre = archivo.replace(".dzn", "")
        datos  = leer_instancia_dzn(_os_gb.path.join("Instaces_MS-CFLP-CI/Public", archivo))
        ref    = _REF_BENCH[nombre]
        incompatibles = _precomputar_incompatibles(
            datos["numero_clientes"], datos["pares_incompatibles"]
        )

        print(f"\n  {nombre}")
        for semilla in _SEMILLAS_BENCH:
            # ── Solución inicial compartida (pool + BL) ────────────────────────
            ab0, asig0, rest0 = seleccionar_solucion_aleatoria_del_pool(datos, _POOL_BENCH, semilla)
            ab0, asig0, rest0 = mejorar_solucion_con_busqueda_local(
                ab0, asig0, rest0,
                datos["numero_instalaciones"], datos["numero_clientes"],
                datos["capacidad_instalacion"], datos["costo_fijo_instalacion"],
                datos["demanda_cliente"], datos["costo_envio"], incompatibles,
            )

            # ── GVNS base (VND fijo N1→N2→N3) ──────────────────────────────────
            t0 = _tgb.perf_counter()
            costo_base = _ejecutar_gvns_iter_vnd(
                datos, list(ab0), list(asig0), list(rest0),
                _K_MAX_BENCH, _N_ITER_BENCH, semilla,
            )
            t_base   = _tgb.perf_counter() - t0
            gap_base = 100 * (costo_base - ref) / ref
            _gb_res[nombre]["GVNS"]["gaps"].append(gap_base)
            _gb_res[nombre]["GVNS"]["t"].append(t_base)

            # ── GVNS + Q-Learning (N1 + N2 + N3, selección adaptativa) ────────
            t0 = _tgb.perf_counter()
            costo_ql, agente = _ejecutar_gvns_iter_ql(
                datos, list(ab0), list(asig0), list(rest0),
                _K_MAX_BENCH, _N_ITER_BENCH, semilla,
            )
            t_ql   = _tgb.perf_counter() - t0
            gap_ql = 100 * (costo_ql - ref) / ref
            _gb_res[nombre]["GVNS+QL"]["gaps"].append(gap_ql)
            _gb_res[nombre]["GVNS+QL"]["t"].append(t_ql)

            speedup = 100 * (t_base - t_ql) / t_base
            print(f"    seed={semilla}  GVNS: {gap_base:+.2f}% {t_base:.1f}s  "
                  f"QL: {gap_ql:+.2f}% {t_ql:.1f}s  Δgap={gap_ql-gap_base:+.2f}pp  "
                  f"speedup={speedup:+.0f}%")

    # ── Tabla resumen ─────────────────────────────────────────────────────────────
    instancias = [a.replace(".dzn", "") for a in _INST_BENCH]
    print(f"\n{'='*76}")
    print(f"  {'':<10}  {'── GVNS base ──':^22}  {'── GVNS+QL ──':^22}  {'Δgap':>6}  {'spd':>5}")
    print(f"  {'Inst.':<10}  {'gap%':>7}  {'σ':>5}  {'t(s)':>5}  "
          f"{'gap%':>7}  {'σ':>5}  {'t(s)':>5}  {'(pp)':>6}  {'(%)':>5}")
    print(f"  {'-'*76}")

    all_base, all_ql = [], []
    for nombre in instancias:
        bg = np.mean(_gb_res[nombre]["GVNS"]["gaps"])
        bs = np.std(_gb_res[nombre]["GVNS"]["gaps"])
        bt = np.mean(_gb_res[nombre]["GVNS"]["t"])
        qg = np.mean(_gb_res[nombre]["GVNS+QL"]["gaps"])
        qs = np.std(_gb_res[nombre]["GVNS+QL"]["gaps"])
        qt = np.mean(_gb_res[nombre]["GVNS+QL"]["t"])
        spd = 100 * (bt - qt) / bt
        all_base.append(bg); all_ql.append(qg)
        print(f"  {nombre:<10}  {bg:>+7.2f}  {bs:>5.2f}  {bt:>5.1f}  "
              f"{qg:>+7.2f}  {qs:>5.2f}  {qt:>5.1f}  {qg-bg:>+6.2f}  {spd:>+5.0f}")

    print(f"  {'-'*76}")
    mb = np.mean(all_base); mq = np.mean(all_ql)
    print(f"  {'Media':<10}  {mb:>+7.2f}  {'':<5}  {'':<5}  {mq:>+7.2f}  {'':<5}  {'':<5}  {mq-mb:>+6.2f}")
    print(f"  {'='*76}")
    print(f"\n  Lectura: la calidad (gap) de GVNS+QL iguala a la base dentro del ruido")
    print(f"  (Δgap ≈ 0), mientras que el speedup positivo refleja el ahorro de las")
    print(f"  pasadas de N2/N3 que el agente aprende a omitir. Política aprendida:")
    print(f"    {agente.resumen()}")
else:
    print("Benchmark GVNS+VND vs GVNS+QL omitido (_EJECUTAR_BENCHMARK_GVNS_QL = False)")


Benchmark GVNS+VND vs GVNS+QL omitido (_EJECUTAR_BENCHMARK_GVNS_QL = False)


## 4. Enfriamiento Simulado + ML

SA acepta movimientos de empeoramiento con probabilidad e^{-Δ/T}, donde la temperatura T
decrece multiplicando por α en cada bloque de movimientos. Al principio, la temperatura alta
permite exploración amplia; al final la concentra. La dificultad está en elegir el ritmo de
descenso: con α demasiado bajo la temperatura cae deprisa y el algoritmo queda atrapado; con
α demasiado alto converge lento.

`ControladorTemperaturaML` aborda este problema ajustando α en tiempo de ejecución según la
tasa de aceptación de movimientos recientes.

### Algoritmo base

Construye la solución inicial con GRASP (α = 0.3) seguido de búsqueda local. En cada paso
del bucle principal se elige un cliente al azar, se propone reasignarlo a una instalación
factible diferente y se calcula el delta de coste.

In [360]:
def recocido_simulado(
    datos: DatosCFLP,
    n_movimientos: int = 5_000,
    ratio_inicial: float = 0.999,
    semilla: int = 0,
    tiempo_limite_s: float | None = None,
) -> ResultadoBenchmark:
    """Enfriamiento Simulado con ratio de enfriamiento fijo.

    Cada iteración elige al azar uno de los tres tipos de movimiento del GVNS
    —N1 (reasignar un cliente), N2 (cerrar una instalación abierta) o N3
    (abrir una instalación cerrada)— y le aplica el criterio de Metropolis.
    Antes SA solo usaba N1: con un único cliente por movimiento nunca puede
    reconsiderar qué instalaciones están abiertas salvo por efecto colateral
    (que el último cliente de una instalación se vaya, o que la primera
    llegada abra una cerrada), algo que en instancias grandes es
    estadísticamente casi imposible de encadenar al azar. N2 y N3 le dan a SA
    la misma capacidad de reestructurar el conjunto de instalaciones abiertas
    que ya tenía la búsqueda local que lo inicializa, con la diferencia de que
    aquí una reestructuración perjudicial también puede aceptarse mientras la
    temperatura lo permita.
    """
    random.seed(semilla)
    t_inicio    = time.perf_counter()
    num_instalaciones = datos["numero_instalaciones"]
    num_clientes      = datos["numero_clientes"]
    capacidad         = datos["capacidad_instalacion"]
    costo_fijo        = datos["costo_fijo_instalacion"]
    demanda           = datos["demanda_cliente"]
    envio             = datos["costo_envio"]
    incompatibles     = _precomputar_incompatibles(num_clientes, datos["pares_incompatibles"])

    abierta, asignada, restante = construir_solucion_greedy_aleatoria(
        num_instalaciones, num_clientes, capacidad, costo_fijo, demanda, envio, 0.3,
        incompatibles,
    )
    abierta, asignada, restante = mejorar_solucion_con_busqueda_local(
        abierta, asignada, restante,
        num_instalaciones, num_clientes, capacidad, costo_fijo, demanda, envio,
        incompatibles,
    )
    costo = calcular_costo_total(abierta, asignada, costo_fijo, demanda, envio)

    mejor_abierta  = list(abierta)
    mejor_asignada = list(asignada)
    mejor_costo    = costo

    clientes_por_instalacion = [set() for _ in range(num_instalaciones)]
    for cliente in range(num_clientes):
        clientes_por_instalacion[asignada[cliente]].add(cliente)

    temperatura = costo * 0.02
    ratio       = ratio_inicial

    for num_iter in range(n_movimientos):
        if tiempo_limite_s is not None and time.perf_counter() - t_inicio >= tiempo_limite_s:
            break

        tipo_movimiento = random.choice(("N1", "N2", "N3"))

        if tipo_movimiento == "N1":
            cliente            = random.randint(0, num_clientes - 1)
            instalacion_origen = asignada[cliente]
            demanda_cliente    = demanda[cliente]
            inc                = incompatibles[cliente]

            instalaciones_prohibidas = {asignada[k] for k in inc}
            candidatas = [
                inst for inst in range(num_instalaciones)
                if inst != instalacion_origen
                and restante[inst] >= demanda_cliente
                and inst not in instalaciones_prohibidas
            ]
            if not candidatas:
                continue
            instalacion_destino = random.choice(candidatas)

            abre_destino  = not abierta[instalacion_destino]
            cierra_origen = len(clientes_por_instalacion[instalacion_origen]) == 1
            delta = _delta_reasignar_cliente(
                cliente, instalacion_origen, instalacion_destino,
                abierta, costo_fijo, demanda, envio, cierra_origen,
            )

            acepta = delta <= 0 or (temperatura > 1e-10 and random.random() < math.exp(-delta / temperatura))
            if acepta:
                restante[instalacion_origen]  += demanda_cliente
                restante[instalacion_destino] -= demanda_cliente
                clientes_por_instalacion[instalacion_origen].discard(cliente)
                clientes_por_instalacion[instalacion_destino].add(cliente)
                asignada[cliente] = instalacion_destino
                if abre_destino:
                    abierta[instalacion_destino] = True
                if cierra_origen:
                    abierta[instalacion_origen] = False
                costo += delta
                if costo < mejor_costo:
                    mejor_costo    = costo
                    mejor_abierta  = list(abierta)
                    mejor_asignada = list(asignada)

        elif tipo_movimiento == "N2":
            abiertas = [i for i in range(num_instalaciones) if abierta[i]]
            if len(abiertas) <= 1:
                continue
            instalacion = random.choice(abiertas)
            propuesta = _evaluar_cierre_instalacion(
                instalacion, abierta, asignada, restante,
                num_instalaciones, num_clientes, costo_fijo, demanda, envio, incompatibles,
            )
            if propuesta is None:
                continue
            delta, reasignacion = propuesta
            # N2 mueve varios clientes a la vez: su delta crece con el tamaño
            # del cierre, mientras que N1 siempre mueve a uno solo. Comparar
            # el delta crudo contra la misma temperatura que N1 sesga la
            # aceptación hacia cierres grandes sin más razón que su tamaño; se
            # normaliza por cliente afectado para que los tres tipos de
            # movimiento compitan en una escala de coste comparable.
            delta_aceptacion = delta / len(reasignacion)
            acepta = delta_aceptacion <= 0 or (temperatura > 1e-10 and random.random() < math.exp(-delta_aceptacion / temperatura))
            if acepta:
                for cliente, nueva_instalacion in reasignacion.items():
                    d = demanda[cliente]
                    restante[instalacion]      += d
                    restante[nueva_instalacion] -= d
                    clientes_por_instalacion[instalacion].discard(cliente)
                    clientes_por_instalacion[nueva_instalacion].add(cliente)
                    asignada[cliente] = nueva_instalacion
                    abierta[nueva_instalacion] = True
                abierta[instalacion] = False
                costo += delta
                if costo < mejor_costo:
                    mejor_costo    = costo
                    mejor_abierta  = list(abierta)
                    mejor_asignada = list(asignada)

        else:  # N3
            cerradas = [i for i in range(num_instalaciones) if not abierta[i]]
            if not cerradas:
                continue
            instalacion = random.choice(cerradas)
            propuesta = _evaluar_apertura_instalacion(
                instalacion, abierta, asignada, restante,
                num_clientes, costo_fijo, demanda, envio, incompatibles,
                [len(s) for s in clientes_por_instalacion],
            )
            if propuesta is None:
                continue
            delta, seleccionados = propuesta
            # Misma normalización que en N2: N3 también mueve varios clientes
            # a la vez.
            delta_aceptacion = delta / len(seleccionados)
            acepta = delta_aceptacion <= 0 or (temperatura > 1e-10 and random.random() < math.exp(-delta_aceptacion / temperatura))
            if acepta:
                origenes_afectados = set()
                for cliente in seleccionados:
                    origen = asignada[cliente]
                    d = demanda[cliente]
                    restante[origen]      += d
                    restante[instalacion] -= d
                    clientes_por_instalacion[origen].discard(cliente)
                    clientes_por_instalacion[instalacion].add(cliente)
                    asignada[cliente] = instalacion
                    origenes_afectados.add(origen)
                abierta[instalacion] = True
                for origen in origenes_afectados:
                    if not clientes_por_instalacion[origen]:
                        abierta[origen] = False
                costo += delta
                if costo < mejor_costo:
                    mejor_costo    = costo
                    mejor_abierta  = list(abierta)
                    mejor_asignada = list(asignada)

        if (num_iter + 1) % 200 == 0:
            temperatura *= ratio
            if temperatura < 1e-10:
                temperatura = mejor_costo * 5e-4

    return ResultadoBenchmark(
        nombre="SA",
        costo_final=mejor_costo,
        tiempo_total_s=time.perf_counter() - t_inicio,
    )


### Componente ML: SelectorRatioUCB

El SA base (celda anterior) usa un ratio de enfriamiento **fijo**
(`ratio_inicial`). La alternativa natural es dejar que un agente decida el
ratio sobre la marcha, igual que se hizo con el selector de $\alpha$ en GRASP y
con Q-learning en GVNS. Aquí se usa un bandido multi-brazo **UCB1**: cada
brazo es uno de los ratios candidatos `{0.90, 0.95, 0.99, 0.995, 0.999}`, y la
recompensa de un brazo es la mejora relativa de coste durante el intervalo en
el que estuvo activo.


In [361]:
class SelectorRatioUCB:
    """Selecciona el ratio de enfriamiento mediante UCB1 (bandido multi-brazo).

    Se probaron variantes descontadas (D-UCB) pensadas para el caso no
    estacionario, con la hipótesis de que el ratio conveniente cambia a lo
    largo de la ejecución. Empíricamente el descuento no ayudó: en un barrido
    sobre \(\gamma \in \{0.9, 0.95, 0.999, 0.9995\}\) y varios intervalos de
    decisión, el factor de descuento apenas cambiaba el resultado, mientras
    que la frecuencia con la que se recalcula la recompensa (`intervalo_ajuste`
    en `recocido_simulado_ml`) sí lo hacía de forma drástica: con intervalos
    cortos (100 movimientos) la recompensa —la mejora relativa de coste en ese
    intervalo— es demasiado ruidosa para distinguir un ratio de otro, sin
    importar si las estadísticas se descuentan o no. Alargando el intervalo a
    2000 movimientos el ruido se reduce lo suficiente para que el propio UCB1
    (sin descuento) supere al enfriamiento fijo. Por eso aquí se usa UCB1
    clásico: es más simple y, con el intervalo adecuado, funciona igual o
    mejor que la variante descontada.
    """

    RATIOS = [0.90, 0.95, 0.99, 0.995, 0.999]

    def __init__(self, c: float = 1.0) -> None:
        self.c                       = c
        self.conteo                  = [0] * len(self.RATIOS)
        self.suma_recompensa         = [0.0] * len(self.RATIOS)
        self.total_tiradas           = 0
        self.ultimo_brazo: int | None = None
        # UCB no entrena un modelo estadístico: se dejan a 0 por compatibilidad
        # con el resto de selectores ML del cuaderno (mismo campo en ResultadoBenchmark).
        self.tiempo_entrenamiento_s = 0.0
        self.tiempo_inferencia_s    = 0.0

    def seleccionar_ratio(self) -> float:
        """Elige un brazo (ratio) por UCB1 y lo devuelve."""
        t0 = time.perf_counter()
        no_probados = [i for i, n in enumerate(self.conteo) if n == 0]
        if no_probados:
            indice = no_probados[0]  # agota cada brazo una vez antes de usar UCB
        else:
            ucb = [
                self.suma_recompensa[i] / self.conteo[i]
                + self.c * math.sqrt(2 * math.log(self.total_tiradas) / self.conteo[i])
                for i in range(len(self.RATIOS))
            ]
            indice = max(range(len(self.RATIOS)), key=lambda i: ucb[i])
        self.ultimo_brazo = indice
        self.tiempo_inferencia_s += time.perf_counter() - t0
        return self.RATIOS[indice]

    def registrar_recompensa(self, recompensa: float) -> None:
        """Acredita la recompensa del intervalo que acaba de terminar al
        brazo que estuvo activo durante ese intervalo."""
        indice = self.ultimo_brazo
        self.conteo[indice]          += 1
        self.suma_recompensa[indice] += recompensa
        self.total_tiradas           += 1


### SA con control adaptativo de temperatura

`recocido_simulado_ml` es idéntico al SA base —mismos tres movimientos
N1/N2/N3, mismo criterio de Metropolis— salvo que el ratio de enfriamiento no
es fijo: cada `intervalo_decidir` movimientos, `SelectorRatioUCB` elige uno
nuevo en función de lo bien que ha funcionado cada ratio hasta ahora.

Medido de forma limpia (mismo presupuesto de tiempo que el SA base), **el
bandido no aporta ganancia de calidad sobre un ratio fijo bien elegido**: en
`wlp01` queda entre $1$ y $2$ puntos porcentuales por detrás, y en `wlp03` la
diferencia ya es de apenas $0.3$ puntos porcentuales, dentro del ruido entre
semillas. La brecha se explica por el coste de la propia exploración del
bandido (probar ratios peores para aprender cuál es mejor) más que por una
limitación de fondo, y se reduce cuanto mayor es la instancia. Esto es
coherente con lo observado en GVNS+QL: cuando la búsqueda subyacente ya está
bien diseñada —aquí, gracias a que el propio SA explora N1, N2 y N3 con
Metropolis en cada movimiento—, el calendario de enfriamiento tiene poca
estructura explotable que aprender. Se incluye el controlador UCB1 por
completitud y para documentar este resultado negativo honesto; la
contribución real de esta sección es haber extendido el SA a los tres
vecindarios del GVNS, no el bandido.


In [362]:
def recocido_simulado_ml(
    datos: DatosCFLP,
    n_movimientos: int = 5_000,
    intervalo_enfriar: int = 200,
    intervalo_decidir: int = 2_000,
    semilla: int = 0,
    tiempo_limite_s: float | None = None,
) -> ResultadoBenchmark:
    """SA con los mismos tres movimientos (N1/N2/N3) que `recocido_simulado`,
    pero con el ratio de enfriamiento elegido por un bandido UCB1 en lugar de
    fijo.

    El diseño anterior (perturbar cerrando varias instalaciones + reoptimizar
    con búsqueda local completa + Metropolis sobre el óptimo local resultante)
    tenía sentido cuando el SA base solo movía un cliente por paso y por tanto
    no podía reestructurar qué instalaciones estaban abiertas: esa era la
    única forma de dar saltos grandes. Ahora que `recocido_simulado` ya explora
    N1/N2/N3 con Metropolis sobre cada movimiento individual, ese rediseño
    quedó por debajo del propio SA base (mismo presupuesto de tiempo, pero
    unas pocas iteraciones caras con reoptimización completa exploran mucho
    menos que millones de movimientos baratos con temple bien calibrado). Aquí
    se vuelve a la misma búsqueda por movimientos que el SA base, cambiando
    únicamente el enfriamiento: en vez de un ratio fijo, un bandido UCB1 elige
    entre `{0.90, 0.95, 0.99, 0.995, 0.999}` cada `intervalo_decidir`
    movimientos (recompensa = mejora relativa de `mejor_costo` en ese
    intervalo). `intervalo_decidir=2000` porque con intervalos más cortos la
    recompensa es demasiado ruidosa para que el bandido distinga ratios
    buenos de malos (ver `SelectorRatioUCB`).
    """
    random.seed(semilla)
    t_inicio    = time.perf_counter()
    num_instalaciones = datos["numero_instalaciones"]
    num_clientes      = datos["numero_clientes"]
    capacidad         = datos["capacidad_instalacion"]
    costo_fijo        = datos["costo_fijo_instalacion"]
    demanda           = datos["demanda_cliente"]
    envio             = datos["costo_envio"]
    incompatibles     = _precomputar_incompatibles(num_clientes, datos["pares_incompatibles"])

    abierta, asignada, restante = construir_solucion_greedy_aleatoria(
        num_instalaciones, num_clientes, capacidad, costo_fijo, demanda, envio, 0.3,
        incompatibles,
    )
    abierta, asignada, restante = mejorar_solucion_con_busqueda_local(
        abierta, asignada, restante,
        num_instalaciones, num_clientes, capacidad, costo_fijo, demanda, envio,
        incompatibles,
    )
    costo = calcular_costo_total(abierta, asignada, costo_fijo, demanda, envio)

    mejor_abierta  = list(abierta)
    mejor_asignada = list(asignada)
    mejor_costo    = costo

    clientes_por_instalacion = [set() for _ in range(num_instalaciones)]
    for cliente in range(num_clientes):
        clientes_por_instalacion[asignada[cliente]].add(cliente)

    temperatura = costo * 0.02
    selector    = SelectorRatioUCB()
    ratio       = selector.seleccionar_ratio()
    mejor_costo_intervalo = mejor_costo

    num_iter = 0
    while num_iter < n_movimientos:
        if tiempo_limite_s is not None and time.perf_counter() - t_inicio >= tiempo_limite_s:
            break

        tipo_movimiento = random.choice(("N1", "N2", "N3"))

        if tipo_movimiento == "N1":
            cliente            = random.randint(0, num_clientes - 1)
            instalacion_origen = asignada[cliente]
            demanda_cliente    = demanda[cliente]
            inc                = incompatibles[cliente]

            instalaciones_prohibidas = {asignada[k] for k in inc}
            candidatas = [
                inst for inst in range(num_instalaciones)
                if inst != instalacion_origen
                and restante[inst] >= demanda_cliente
                and inst not in instalaciones_prohibidas
            ]
            if candidatas:
                instalacion_destino = random.choice(candidatas)

                abre_destino  = not abierta[instalacion_destino]
                cierra_origen = len(clientes_por_instalacion[instalacion_origen]) == 1
                delta = _delta_reasignar_cliente(
                    cliente, instalacion_origen, instalacion_destino,
                    abierta, costo_fijo, demanda, envio, cierra_origen,
                )

                acepta = delta <= 0 or (temperatura > 1e-10 and random.random() < math.exp(-delta / temperatura))
                if acepta:
                    restante[instalacion_origen]  += demanda_cliente
                    restante[instalacion_destino] -= demanda_cliente
                    clientes_por_instalacion[instalacion_origen].discard(cliente)
                    clientes_por_instalacion[instalacion_destino].add(cliente)
                    asignada[cliente] = instalacion_destino
                    if abre_destino:
                        abierta[instalacion_destino] = True
                    if cierra_origen:
                        abierta[instalacion_origen] = False
                    costo += delta
                    if costo < mejor_costo:
                        mejor_costo    = costo
                        mejor_abierta  = list(abierta)
                        mejor_asignada = list(asignada)

        elif tipo_movimiento == "N2":
            abiertas = [i for i in range(num_instalaciones) if abierta[i]]
            if len(abiertas) > 1:
                instalacion = random.choice(abiertas)
                propuesta = _evaluar_cierre_instalacion(
                    instalacion, abierta, asignada, restante,
                    num_instalaciones, num_clientes, costo_fijo, demanda, envio, incompatibles,
                )
                if propuesta is not None:
                    delta, reasignacion = propuesta
                    delta_aceptacion = delta / len(reasignacion)
                    acepta = delta_aceptacion <= 0 or (temperatura > 1e-10 and random.random() < math.exp(-delta_aceptacion / temperatura))
                    if acepta:
                        for cliente, nueva_instalacion in reasignacion.items():
                            d = demanda[cliente]
                            restante[instalacion]      += d
                            restante[nueva_instalacion] -= d
                            clientes_por_instalacion[instalacion].discard(cliente)
                            clientes_por_instalacion[nueva_instalacion].add(cliente)
                            asignada[cliente] = nueva_instalacion
                            abierta[nueva_instalacion] = True
                        abierta[instalacion] = False
                        costo += delta
                        if costo < mejor_costo:
                            mejor_costo    = costo
                            mejor_abierta  = list(abierta)
                            mejor_asignada = list(asignada)

        else:  # N3
            cerradas = [i for i in range(num_instalaciones) if not abierta[i]]
            if cerradas:
                instalacion = random.choice(cerradas)
                propuesta = _evaluar_apertura_instalacion(
                    instalacion, abierta, asignada, restante,
                    num_clientes, costo_fijo, demanda, envio, incompatibles,
                    [len(s) for s in clientes_por_instalacion],
                )
                if propuesta is not None:
                    delta, seleccionados = propuesta
                    delta_aceptacion = delta / len(seleccionados)
                    acepta = delta_aceptacion <= 0 or (temperatura > 1e-10 and random.random() < math.exp(-delta_aceptacion / temperatura))
                    if acepta:
                        origenes_afectados = set()
                        for cliente in seleccionados:
                            origen = asignada[cliente]
                            d = demanda[cliente]
                            restante[origen]      += d
                            restante[instalacion] -= d
                            clientes_por_instalacion[origen].discard(cliente)
                            clientes_por_instalacion[instalacion].add(cliente)
                            asignada[cliente] = instalacion
                            origenes_afectados.add(origen)
                        abierta[instalacion] = True
                        for origen in origenes_afectados:
                            if not clientes_por_instalacion[origen]:
                                abierta[origen] = False
                        costo += delta
                        if costo < mejor_costo:
                            mejor_costo    = costo
                            mejor_abierta  = list(abierta)
                            mejor_asignada = list(asignada)

        if (num_iter + 1) % intervalo_enfriar == 0:
            temperatura *= ratio
            if temperatura < 1e-10:
                temperatura = mejor_costo * 5e-4

        if (num_iter + 1) % intervalo_decidir == 0:
            if selector.ultimo_brazo is not None:
                recompensa = max(0.0, (mejor_costo_intervalo - mejor_costo) / mejor_costo_intervalo)
                selector.registrar_recompensa(recompensa)
            ratio = selector.seleccionar_ratio()
            mejor_costo_intervalo = mejor_costo

        num_iter += 1

    return ResultadoBenchmark(
        nombre="SA + UCB1",
        costo_final=mejor_costo,
        tiempo_total_s=time.perf_counter() - t_inicio,
        tiempo_entrenamiento_ml_s=selector.tiempo_entrenamiento_s,
        tiempo_inferencia_ml_s=selector.tiempo_inferencia_s,
        modelo_ml=selector,
    )


### Benchmark SA (movimientos, base) vs SA+UCB1

Compara **SA** por movimientos con ratio de enfriamiento fijo (celda
`sa-base-code`) contra **SA+UCB1**, la misma búsqueda por movimientos N1/N2/N3
pero con el ratio de enfriamiento elegido por un bandido UCB1 en vez de fijo
(celda `04b06824`), sobre `wlp01`–`wlp04`, 5 semillas por instancia.

Los dos algoritmos comparten exactamente el mismo tipo de paso (movimientos
individuales N1/N2/N3 con Metropolis; la única diferencia es cómo se elige el
ratio de enfriamiento), así que un mismo `n_movimientos` les da presupuestos
comparables — a diferencia del rediseño anterior como Búsqueda Local Iterada,
cuyas iteraciones (perturbar + búsqueda local completa) costaban mucho más
que un movimiento y por eso necesitaban igualar el presupuesto por tiempo de
reloj en vez de por conteo. Por eso aquí se iguala directamente por conteo:
ambos reciben el mismo `n_movimientos`, sin medir ni limitar por tiempo.


In [363]:
import os as _os_sab
import time as _tsab
import numpy as np

_EJECUTAR_BENCHMARK_SA_UCB1 = False  # cambia a True para ejecutar este benchmark (lento)

if _EJECUTAR_BENCHMARK_SA_UCB1:

    _INST_SA_BENCH      = ["wlp01.dzn", "wlp02.dzn", "wlp03.dzn", "wlp04.dzn"]
    _REF_SA_BENCH        = {"wlp01": 28716, "wlp02": 52952, "wlp03": 64296, "wlp04": 84633}
    _SEMILLAS_SA_BENCH   = [0, 1, 2, 3, 4]
    _N_MOV_SA_BENCH      = 500_000     # movimientos, igual para SA y SA+UCB1

    _sa_res = {
        nombre: {"SA": {"gaps": [], "t": []},
                 "SA+UCB1": {"gaps": [], "t": []}}
        for nombre in [a.replace(".dzn", "") for a in _INST_SA_BENCH]
    }

    for archivo in _INST_SA_BENCH:
        nombre = archivo.replace(".dzn", "")
        datos  = leer_instancia_dzn(_os_sab.path.join("Instaces_MS-CFLP-CI/Public", archivo))
        ref    = _REF_SA_BENCH[nombre]

        print(f"\n  {nombre}")
        for semilla in _SEMILLAS_SA_BENCH:
            # ── SA base (movimientos, ratio de enfriamiento fijo) ─────────────
            t0 = _tsab.perf_counter()
            r_base = recocido_simulado(datos, n_movimientos=_N_MOV_SA_BENCH, semilla=semilla)
            t_base   = _tsab.perf_counter() - t0
            gap_base = 100 * (r_base.costo_final - ref) / ref
            _sa_res[nombre]["SA"]["gaps"].append(gap_base)
            _sa_res[nombre]["SA"]["t"].append(t_base)

            # ── SA+UCB1, mismo número de MOVIMIENTOS que el SA base ───────────
            t0 = _tsab.perf_counter()
            r_ml = recocido_simulado_ml(
                datos, n_movimientos=_N_MOV_SA_BENCH, semilla=semilla,
            )
            t_ml   = _tsab.perf_counter() - t0
            gap_ml = 100 * (r_ml.costo_final - ref) / ref
            _sa_res[nombre]["SA+UCB1"]["gaps"].append(gap_ml)
            _sa_res[nombre]["SA+UCB1"]["t"].append(t_ml)

            print(f"    seed={semilla}  SA: {gap_base:+.2f}% {t_base:.1f}s  "
                  f"SA+UCB1: {gap_ml:+.2f}% {t_ml:.1f}s  Δgap={gap_ml-gap_base:+.2f}pp")

    # ── Tabla resumen ─────────────────────────────────────────────────────────────
    instancias_sa = [a.replace(".dzn", "") for a in _INST_SA_BENCH]
    print(f"\n{'='*70}")
    print(f"  {'':<10}  {'── SA base ──':^18}  {'── SA+UCB1 ──':^18}  {'Δgap':>6}")
    print(f"  {'Inst.':<10}  {'gap%':>7}  {'σ':>5}  {'t(s)':>5}  "
          f"{'gap%':>7}  {'σ':>5}  {'t(s)':>5}  {'(pp)':>6}")
    print(f"  {'-'*70}")

    all_base, all_ml = [], []
    for nombre in instancias_sa:
        bg = np.mean(_sa_res[nombre]["SA"]["gaps"])
        bs = np.std(_sa_res[nombre]["SA"]["gaps"])
        bt = np.mean(_sa_res[nombre]["SA"]["t"])
        mg = np.mean(_sa_res[nombre]["SA+UCB1"]["gaps"])
        ms = np.std(_sa_res[nombre]["SA+UCB1"]["gaps"])
        mt = np.mean(_sa_res[nombre]["SA+UCB1"]["t"])
        all_base.append(bg); all_ml.append(mg)
        print(f"  {nombre:<10}  {bg:>+7.2f}  {bs:>5.2f}  {bt:>5.1f}  "
              f"{mg:>+7.2f}  {ms:>5.2f}  {mt:>5.1f}  {mg-bg:>+6.2f}")

    print(f"  {'-'*70}")
    mb = np.mean(all_base); mm = np.mean(all_ml)
    print(f"  {'Media':<10}  {mb:>+7.2f}  {'':<5}  {'':<5}  {mm:>+7.2f}  {'':<5}  {'':<5}  {mm-mb:>+6.2f}")
    print(f"  {'='*70}")
else:
    print("Benchmark SA vs SA+UCB1 omitido (_EJECUTAR_BENCHMARK_SA_UCB1 = False)")


Benchmark SA vs SA+UCB1 omitido (_EJECUTAR_BENCHMARK_SA_UCB1 = False)


## 5. ALNS (Adaptive Large Neighborhood Search) + selección adaptativa de operadores

La Búsqueda Tabú por movimientos muestreados (30 clientes × 15 instalaciones por
iteración) queda estructuralmente limitada en instancias grandes: al explorar solo
una fracción fija y pequeña del vecindario en cada paso, necesita cientos de miles
de iteraciones para acercarse a una buena solución, y aun así se estanca muy por
encima de lo que consiguen SA o GVNS.

ALNS (Ropke y Pisinger, 2006) ataca el problema de forma distinta: en vez de mover
un cliente a la vez, en cada iteración **destruye** una parte de la solución
actual —elimina varios clientes de sus instalaciones— y la **repara**
reinsertándolos con una heurística constructiva. Al operar sobre bloques grandes de
la solución en cada paso escapa de mínimos locales que un movimiento individual
nunca alcanzaría, sin necesidad de listas tabú ni de una tenencia que calibrar.

`SelectorOperadorALNS` sustituye la elección uniforme de operador (destrucción y
reparación) por una selección adaptativa: cada operador acumula un peso según qué
tan bien le ha ido, y con el tiempo los operadores más productivos se eligen con
más frecuencia.

### Operadores de destrucción

Cada operador recibe un tamaño `q` (clientes a eliminar) y libera esos clientes de
sus instalaciones actuales, cerrando cualquier instalación que se quede sin
clientes:

- **D1 — Aleatorio**: elimina `q` clientes al azar. El operador más simple; sirve
  de referencia y de diversificación pura.
- **D2 — Peor coste**: elimina los `q` clientes cuyo coste de envío actual es más
  alto. Ataca directamente las asignaciones más caras de la solución.
- **D3 — Relacionado (Shaw removal)**: parte de un cliente semilla al azar y va
  añadiendo, uno a uno, el cliente restante más "relacionado" con el último
  elegido —aquel cuyo coste cruzado de servirse mutuamente desde la instalación
  del otro es menor—. Agrupa clientes que tendría sentido reasignar juntos.
- **D4 — Cierre de instalación**: cierra una instalación abierta al azar y libera
  a todos sus clientes de golpe. Es la única forma de que ALNS reconsidere
  directamente qué instalaciones están abiertas, más que reordenar clientes entre
  las ya abiertas.

In [364]:
def _liberar_cliente_alns(cliente, asignada, restante, abierta, clientes_por_instalacion, demanda):
    """Desasigna `cliente` de su instalación actual, liberando su capacidad y
    cerrando la instalación si se queda sin clientes."""
    origen = asignada[cliente]
    d = demanda[cliente]
    restante[origen] += d
    clientes_por_instalacion[origen].discard(cliente)
    asignada[cliente] = -1
    if not clientes_por_instalacion[origen]:
        abierta[origen] = False


def _destruir_aleatorio(asignada, restante, abierta, clientes_por_instalacion, demanda, q, num_clientes):
    """D1: elimina q clientes al azar."""
    candidatos = [j for j in range(num_clientes) if asignada[j] != -1]
    q = min(q, len(candidatos))
    eliminados = random.sample(candidatos, q)
    for j in eliminados:
        _liberar_cliente_alns(j, asignada, restante, abierta, clientes_por_instalacion, demanda)
    return eliminados


def _destruir_peor_coste(asignada, restante, abierta, clientes_por_instalacion, demanda, envio, q, num_clientes):
    """D2: elimina los q clientes cuyo coste de envío actual es más alto (asignaciones caras)."""
    candidatos = [j for j in range(num_clientes) if asignada[j] != -1]
    costes = [(envio[asignada[j]][j] * demanda[j], j) for j in candidatos]
    costes.sort(reverse=True)
    q = min(q, len(costes))
    eliminados = [j for _, j in costes[:q]]
    for j in eliminados:
        _liberar_cliente_alns(j, asignada, restante, abierta, clientes_por_instalacion, demanda)
    return eliminados


def _destruir_relacionado(asignada, restante, abierta, clientes_por_instalacion, demanda, envio, q, num_clientes):
    """D3 (Shaw removal): parte de un cliente semilla al azar y añade, uno a uno,
    el cliente restante más 'relacionado' con el último elegido: aquel cuyo coste
    cruzado de servirse mutuamente desde la instalación del otro es menor
    (candidatos baratos de agrupar/intercambiar entre sí)."""
    candidatos = [j for j in range(num_clientes) if asignada[j] != -1]
    q = min(q, len(candidatos))
    if q == 0:
        return []
    semilla = random.choice(candidatos)
    elegidos = [semilla]
    restantes = set(candidatos) - {semilla}
    while len(elegidos) < q and restantes:
        ref = elegidos[-1]
        f_ref = asignada[ref]

        def _relacion(k, f_ref=f_ref, ref=ref):
            return envio[f_ref][k] + envio[asignada[k]][ref]

        siguiente = min(restantes, key=_relacion)
        elegidos.append(siguiente)
        restantes.discard(siguiente)
    for j in elegidos:
        _liberar_cliente_alns(j, asignada, restante, abierta, clientes_por_instalacion, demanda)
    return elegidos


def _destruir_instalacion(asignada, restante, abierta, clientes_por_instalacion, demanda, num_instalaciones):
    """D4: cierra una instalación abierta al azar y libera a todos sus clientes."""
    abiertas = [i for i in range(num_instalaciones) if abierta[i]]
    if not abiertas:
        return []
    instalacion = random.choice(abiertas)
    eliminados = list(clientes_por_instalacion[instalacion])
    for j in eliminados:
        _liberar_cliente_alns(j, asignada, restante, abierta, clientes_por_instalacion, demanda)
    return eliminados

### Operadores de reparación

Reciben la lista de clientes eliminados y los reinsertan uno a uno en instalaciones
factibles (con capacidad y sin incompatibilidades):

- **R1 — Greedy**: procesa los clientes eliminados en orden aleatorio e inserta
  cada uno en su instalación factible más barata en ese momento.
- **R2 — Arrepentimiento (regret-2)**: para cada cliente eliminado calcula la
  diferencia entre su mejor y su segunda mejor instalación factible (su
  "arrepentimiento" si se pospone), e inserta primero a quien más tiene que
  perder. Es más caro que R1 pero suele producir mejores reparaciones porque no
  deja para el final a los clientes con menos alternativas.

In [365]:
def _mejor_insercion_factible(cliente, asignada, restante, abierta, num_instalaciones, demanda, costo_fijo, envio, incompatibles):
    """Devuelve (coste_insercion, instalacion) de la mejor opción factible para
    `cliente`, o (inf, None) si ninguna instalación tiene capacidad/compatibilidad."""
    d = demanda[cliente]
    inc = incompatibles[cliente]
    mejor_costo, mejor_inst = float("inf"), None
    for i in range(num_instalaciones):
        if restante[i] < d:
            continue
        if any(asignada[k] == i for k in inc):
            continue
        c = envio[i][cliente] * d
        if not abierta[i]:
            c += costo_fijo[i]
        if c < mejor_costo:
            mejor_costo, mejor_inst = c, i
    return mejor_costo, mejor_inst


def _asignar_cliente(cliente, instalacion, asignada, restante, abierta, clientes_por_instalacion, demanda):
    d = demanda[cliente]
    asignada[cliente] = instalacion
    restante[instalacion] -= d
    clientes_por_instalacion[instalacion].add(cliente)
    abierta[instalacion] = True


def _reparar_greedy(eliminados, asignada, restante, abierta, clientes_por_instalacion,
                     num_instalaciones, demanda, costo_fijo, envio, incompatibles):
    """R1: inserta los clientes eliminados, en orden aleatorio, en su mejor
    instalación factible en el momento de insertarlos."""
    orden = list(eliminados)
    random.shuffle(orden)
    for j in orden:
        _, inst = _mejor_insercion_factible(
            j, asignada, restante, abierta, num_instalaciones, demanda, costo_fijo, envio, incompatibles
        )
        if inst is not None:
            _asignar_cliente(j, inst, asignada, restante, abierta, clientes_por_instalacion, demanda)
        # Si ninguna instalación tiene capacidad (posible con incompatibilidades
        # extremas), el cliente queda sin asignar: no debería ocurrir partiendo
        # de una instancia factible, pero no se fuerza para no romper capacidad.


def _reparar_regret2(eliminados, asignada, restante, abierta, clientes_por_instalacion,
                      num_instalaciones, demanda, costo_fijo, envio, incompatibles):
    """R2 (regret-2): calcula, una única vez, el arrepentimiento de posponer a
    cada cliente removido —la diferencia entre su mejor y su segunda mejor
    instalación factible en el estado de capacidad actual— e inserta a todos en
    ese orden (mayor arrepentimiento primero). Si al llegarle el turno a un
    cliente su mejor opción calculada ya no tiene hueco (porque otro cliente se
    lo ocupó antes), recalcula sobre la marcha solo para ese cliente. Evita
    recalcular el arrepentimiento de todos los pendientes tras cada inserción
    (coste O(q·m) en vez de O(q²·m), decisivo con q~100 en instancias grandes).
    """

    def _mejor_y_segunda(j):
        d = demanda[j]
        inc = incompatibles[j]
        mejor_c, mejor_i = float("inf"), None
        segunda_c = float("inf")
        for i in range(num_instalaciones):
            if restante[i] < d or any(asignada[k] == i for k in inc):
                continue
            c = envio[i][j] * d + (0 if abierta[i] else costo_fijo[i])
            if c < mejor_c:
                segunda_c = mejor_c
                mejor_c, mejor_i = c, i
            elif c < segunda_c:
                segunda_c = c
        return mejor_c, mejor_i, segunda_c

    orden = []
    for j in eliminados:
        mejor_c, mejor_i, segunda_c = _mejor_y_segunda(j)
        regret = (segunda_c - mejor_c) if mejor_i is not None and segunda_c < float("inf") else 0.0
        orden.append((regret, j, mejor_i))
    orden.sort(key=lambda t: t[0], reverse=True)

    for _, j, mejor_i in orden:
        d = demanda[j]
        cabe = mejor_i is not None and restante[mejor_i] >= d and not any(
            asignada[k] == mejor_i for k in incompatibles[j]
        )
        if cabe:
            _asignar_cliente(j, mejor_i, asignada, restante, abierta, clientes_por_instalacion, demanda)
        else:
            # La opción calculada ya no es factible (otro cliente ocupó el
            # hueco): recalcular solo para este cliente.
            _, inst = _mejor_insercion_factible(
                j, asignada, restante, abierta, num_instalaciones, demanda, costo_fijo, envio, incompatibles
            )
            if inst is not None:
                _asignar_cliente(j, inst, asignada, restante, abierta, clientes_por_instalacion, demanda)

### Algoritmo base: ALNS con selección uniforme de operadores

Cada iteración parte de la solución actual y aplica tres pasos: elegir un tamaño de
destrucción `q` (entre el 5 % y el 20 % de los clientes), destruir `q` clientes con
uno de los cuatro operadores anteriores, y repararlos con uno de los dos
operadores constructivos. La solución reparada se acepta con el criterio de
Metropolis (mismo mecanismo que en el capítulo de SA, con calendario de
enfriamiento geométrico fijo — aquí no hay componente ML en la temperatura, solo
en la elección de operador). El algoritmo base elige el operador de destrucción y
el de reparación **uniformemente al azar** en cada iteración; la variante con ML
(más abajo) los elige según pesos aprendidos.

In [366]:
def _aplicar_destroy(tipo, asignada, restante, abierta, clientes_por_instalacion,
                      demanda, envio, costo_fijo, q, num_clientes, num_instalaciones):
    if tipo == 0:
        return _destruir_aleatorio(asignada, restante, abierta, clientes_por_instalacion, demanda, q, num_clientes)
    if tipo == 1:
        return _destruir_peor_coste(asignada, restante, abierta, clientes_por_instalacion, demanda, envio, q, num_clientes)
    if tipo == 2:
        return _destruir_relacionado(asignada, restante, abierta, clientes_por_instalacion, demanda, envio, q, num_clientes)
    return _destruir_instalacion(asignada, restante, abierta, clientes_por_instalacion, demanda, num_instalaciones)


def _aplicar_repair(tipo, eliminados, asignada, restante, abierta, clientes_por_instalacion,
                     num_instalaciones, demanda, costo_fijo, envio, incompatibles):
    if tipo == 0:
        _reparar_greedy(eliminados, asignada, restante, abierta, clientes_por_instalacion,
                         num_instalaciones, demanda, costo_fijo, envio, incompatibles)
    else:
        _reparar_regret2(eliminados, asignada, restante, abierta, clientes_por_instalacion,
                          num_instalaciones, demanda, costo_fijo, envio, incompatibles)


def alns_base(
    datos: DatosCFLP,
    n_iter: int = 2_000,
    ratio_enfriamiento: float = 0.995,
    semilla: int = 0,
    tiempo_limite_s: float | None = None,
) -> ResultadoBenchmark:
    """ALNS con selección uniforme (al azar) de operador de destrucción y de reparación."""
    random.seed(semilla)
    t_inicio = time.perf_counter()
    num_instalaciones = datos["numero_instalaciones"]
    num_clientes      = datos["numero_clientes"]
    costo_fijo        = datos["costo_fijo_instalacion"]
    demanda           = datos["demanda_cliente"]
    envio             = datos["costo_envio"]
    incompatibles     = _precomputar_incompatibles(num_clientes, datos["pares_incompatibles"])

    abierta, asignada, restante = construir_solucion_greedy_aleatoria(
        num_instalaciones, num_clientes, datos["capacidad_instalacion"], costo_fijo, demanda, envio, 0.3,
        incompatibles,
    )
    abierta, asignada, restante = mejorar_solucion_con_busqueda_local(
        abierta, asignada, restante, num_instalaciones, num_clientes,
        datos["capacidad_instalacion"], costo_fijo, demanda, envio, incompatibles,
    )
    costo = calcular_costo_total(abierta, asignada, costo_fijo, demanda, envio)
    mejor_costo = costo

    clientes_por_instalacion = [set() for _ in range(num_instalaciones)]
    for j in range(num_clientes):
        clientes_por_instalacion[asignada[j]].add(j)

    q_min = max(3, num_clientes // 20)
    q_max = max(q_min + 1, num_clientes // 5)
    temperatura = costo * 0.02

    for _ in range(n_iter):
        if tiempo_limite_s is not None and time.perf_counter() - t_inicio >= tiempo_limite_s:
            break

        ab_prev, asig_prev, rest_prev = list(abierta), list(asignada), list(restante)
        cpi_prev = [set(s) for s in clientes_por_instalacion]

        tipo_destroy = random.randint(0, 3)
        tipo_repair  = random.randint(0, 1)
        q = random.randint(q_min, q_max)

        eliminados = _aplicar_destroy(
            tipo_destroy, asignada, restante, abierta, clientes_por_instalacion,
            demanda, envio, costo_fijo, q, num_clientes, num_instalaciones,
        )
        _aplicar_repair(
            tipo_repair, eliminados, asignada, restante, abierta, clientes_por_instalacion,
            num_instalaciones, demanda, costo_fijo, envio, incompatibles,
        )

        costo_nuevo = calcular_costo_total(abierta, asignada, costo_fijo, demanda, envio)
        delta = costo_nuevo - costo

        acepta = delta <= 0 or (temperatura > 1e-10 and random.random() < math.exp(-delta / temperatura))
        if acepta:
            costo = costo_nuevo
            if costo < mejor_costo:
                mejor_costo = costo
        else:
            abierta, asignada, restante = ab_prev, asig_prev, rest_prev
            clientes_por_instalacion = cpi_prev

        temperatura *= ratio_enfriamiento
        if temperatura < 1e-10:
            temperatura = mejor_costo * 5e-4

    return ResultadoBenchmark(
        nombre="ALNS",
        costo_final=mejor_costo,
        tiempo_total_s=time.perf_counter() - t_inicio,
    )

### Componente ML: selección adaptativa de operadores

`SelectorOperadorALNS` mantiene un peso por operador (4 de destrucción, 2 de
reparación) y elige cada uno por ruleta: la probabilidad de elegir un operador es
proporcional a su peso frente al resto. Cada `tamano_segmento` iteraciones (un
"segmento") se acumula, por operador usado, una puntuación según el resultado de
esa iteración:

- **33 puntos** si produjo un nuevo mejor global.
- **20 puntos** si mejoró la solución actual (sin llegar a mejor global).
- **9 puntos** si se aceptó aunque no mejorara (diversificación útil).
- **0 puntos** si se rechazó.

Al final del segmento, el peso de cada operador se actualiza como una media
exponencial entre su peso anterior y su rendimiento medio en el segmento
(puntuación acumulada entre veces usado), y los contadores se reinician para el
siguiente segmento. Es el mismo esquema de pesos adaptativos por ruleta que
introdujeron Ropke y Pisinger (2006) para ALNS, y guarda cierto parecido de
espíritu con el bandido UCB1 del capítulo de SA: ambos reparten más intentos a
las opciones que mejor han rendido sin descartar del todo a las demás, aunque
aquí el mecanismo de actualización es un promedio con memoria en vez de una cota
de confianza.

In [367]:
class SelectorOperadorALNS:
    """Selección adaptativa de operadores de destrucción y reparación mediante
    pesos actualizados por segmentos, al estilo de Ropke y Pisinger (2006).

    Cada operador (4 de destrucción, 2 de reparación) tiene un peso; la
    probabilidad de elegirlo en una iteración es proporcional a su peso
    (selección por ruleta). Durante un segmento de `tamano_segmento`
    iteraciones se acumula, por operador usado, una puntuación según el
    resultado de esa iteración (nuevo óptimo global, mejora del actual, o
    ninguna de las dos --- aceptada sin mejorar o rechazada, que no reciben
    recompensa). Al final del segmento los pesos se actualizan como una media
    exponencial entre el peso anterior y el rendimiento medio (puntuación/usos)
    observado en el segmento, y las puntuaciones se reinician.
    """

    SIGMA_MEJOR_GLOBAL = 33.0
    SIGMA_MEJORA       = 20.0
    # No se premia "aceptada sin mejorar" (probado con 9.0, como en Ropke &
    # Pisinger, y empeoró de forma consistente en las 4 instancias de prueba:
    # +1.03pp de media). Con temperatura alta casi cualquier movimiento se
    # acepta, así que esa señal es puro ruido en la fase inicial de la
    # búsqueda y no aporta información sobre qué operador es mejor.

    def __init__(self, n_destroy: int = 4, n_repair: int = 2,
                 tamano_segmento: int = 100, factor_reaccion: float = 0.2) -> None:
        self.tamano_segmento = tamano_segmento
        self.factor_reaccion = factor_reaccion
        self.pesos_destroy   = [1.0] * n_destroy
        self.pesos_repair    = [1.0] * n_repair
        self._score_destroy  = [0.0] * n_destroy
        self._usos_destroy   = [0] * n_destroy
        self._score_repair   = [0.0] * n_repair
        self._usos_repair    = [0] * n_repair
        self._iter_segmento  = 0
        self.tiempo_entrenamiento_s: float = 0.0
        self.tiempo_inferencia_s: float    = 0.0

    def _elegir(self, pesos: list[float]) -> int:
        t0 = time.perf_counter()
        total = sum(pesos)
        r = random.random() * total
        acumulado = 0.0
        elegido = len(pesos) - 1
        for i, w in enumerate(pesos):
            acumulado += w
            if r <= acumulado:
                elegido = i
                break
        self.tiempo_inferencia_s += time.perf_counter() - t0
        return elegido

    def elegir_destroy(self) -> int:
        return self._elegir(self.pesos_destroy)

    def elegir_repair(self) -> int:
        return self._elegir(self.pesos_repair)

    def registrar(self, tipo_destroy: int, tipo_repair: int, es_mejor_global: bool,
                  mejora_actual: bool) -> None:
        if es_mejor_global:
            sigma = self.SIGMA_MEJOR_GLOBAL
        elif mejora_actual:
            sigma = self.SIGMA_MEJORA
        else:
            sigma = 0.0

        self._score_destroy[tipo_destroy] += sigma
        self._usos_destroy[tipo_destroy]  += 1
        self._score_repair[tipo_repair]   += sigma
        self._usos_repair[tipo_repair]    += 1
        self._iter_segmento += 1

        if self._iter_segmento >= self.tamano_segmento:
            t0 = time.perf_counter()
            r = self.factor_reaccion
            for i in range(len(self.pesos_destroy)):
                if self._usos_destroy[i] > 0:
                    rendimiento = self._score_destroy[i] / self._usos_destroy[i]
                    self.pesos_destroy[i] = (1 - r) * self.pesos_destroy[i] + r * rendimiento
            for i in range(len(self.pesos_repair)):
                if self._usos_repair[i] > 0:
                    rendimiento = self._score_repair[i] / self._usos_repair[i]
                    self.pesos_repair[i] = (1 - r) * self.pesos_repair[i] + r * rendimiento
            self._score_destroy = [0.0] * len(self.pesos_destroy)
            self._usos_destroy  = [0] * len(self.pesos_destroy)
            self._score_repair  = [0.0] * len(self.pesos_repair)
            self._usos_repair   = [0] * len(self.pesos_repair)
            self._iter_segmento = 0
            self.tiempo_entrenamiento_s += time.perf_counter() - t0

### ALNS con selección adaptativa

Idéntico al algoritmo base salvo en un punto: en vez de sortear el operador de
destrucción y el de reparación de forma uniforme, se consulta a
`SelectorOperadorALNS` (selección por ruleta según los pesos vigentes) y se le
registra el resultado de cada iteración para que ajuste esos pesos por segmentos.

In [368]:
def alns_ml(
    datos: DatosCFLP,
    n_iter: int = 2_000,
    ratio_enfriamiento: float = 0.995,
    tamano_segmento: int | None = None,
    factor_reaccion: float = 0.2,
    semilla: int = 0,
    tiempo_limite_s: float | None = None,
) -> ResultadoBenchmark:
    """ALNS con selección adaptativa de operadores vía SelectorOperadorALNS.

    tamano_segmento=None (por defecto) lo escala como max(20, n_iter // 100):
    con un tamaño fijo de 100, presupuestos pequeños (p. ej. n_iter=2000)
    gastaban el primer 5% de toda la búsqueda sin ajustar los pesos ni una
    sola vez; escalarlo evitó esa pérdida de presupuesto sin cambiar nada a
    partir de n_iter=10 000 (donde 10000//100 ya coincide con el valor fijo
    anterior).
    """
    if tamano_segmento is None:
        tamano_segmento = max(20, n_iter // 100)
    random.seed(semilla)
    t_inicio = time.perf_counter()
    num_instalaciones = datos["numero_instalaciones"]
    num_clientes      = datos["numero_clientes"]
    costo_fijo        = datos["costo_fijo_instalacion"]
    demanda           = datos["demanda_cliente"]
    envio             = datos["costo_envio"]
    incompatibles     = _precomputar_incompatibles(num_clientes, datos["pares_incompatibles"])

    abierta, asignada, restante = construir_solucion_greedy_aleatoria(
        num_instalaciones, num_clientes, datos["capacidad_instalacion"], costo_fijo, demanda, envio, 0.3,
        incompatibles,
    )
    abierta, asignada, restante = mejorar_solucion_con_busqueda_local(
        abierta, asignada, restante, num_instalaciones, num_clientes,
        datos["capacidad_instalacion"], costo_fijo, demanda, envio, incompatibles,
    )
    costo = calcular_costo_total(abierta, asignada, costo_fijo, demanda, envio)
    mejor_costo = costo

    clientes_por_instalacion = [set() for _ in range(num_instalaciones)]
    for j in range(num_clientes):
        clientes_por_instalacion[asignada[j]].add(j)

    q_min = max(3, num_clientes // 20)
    q_max = max(q_min + 1, num_clientes // 5)
    temperatura = costo * 0.02
    selector = SelectorOperadorALNS(tamano_segmento=tamano_segmento, factor_reaccion=factor_reaccion)

    for _ in range(n_iter):
        if tiempo_limite_s is not None and time.perf_counter() - t_inicio >= tiempo_limite_s:
            break

        ab_prev, asig_prev, rest_prev = list(abierta), list(asignada), list(restante)
        cpi_prev = [set(s) for s in clientes_por_instalacion]

        tipo_destroy = selector.elegir_destroy()
        tipo_repair  = selector.elegir_repair()
        q = random.randint(q_min, q_max)

        eliminados = _aplicar_destroy(
            tipo_destroy, asignada, restante, abierta, clientes_por_instalacion,
            demanda, envio, costo_fijo, q, num_clientes, num_instalaciones,
        )
        _aplicar_repair(
            tipo_repair, eliminados, asignada, restante, abierta, clientes_por_instalacion,
            num_instalaciones, demanda, costo_fijo, envio, incompatibles,
        )

        costo_nuevo = calcular_costo_total(abierta, asignada, costo_fijo, demanda, envio)
        delta = costo_nuevo - costo

        acepta = delta <= 0 or (temperatura > 1e-10 and random.random() < math.exp(-delta / temperatura))
        es_mejor_global = False
        if acepta:
            mejora_actual = costo_nuevo < costo
            costo = costo_nuevo
            if costo < mejor_costo:
                mejor_costo = costo
                es_mejor_global = True
        else:
            mejora_actual = False
            abierta, asignada, restante = ab_prev, asig_prev, rest_prev
            clientes_por_instalacion = cpi_prev

        selector.registrar(tipo_destroy, tipo_repair, es_mejor_global, mejora_actual)

        temperatura *= ratio_enfriamiento
        if temperatura < 1e-10:
            temperatura = mejor_costo * 5e-4

    return ResultadoBenchmark(
        nombre="ALNS + Adaptativo",
        costo_final=mejor_costo,
        tiempo_total_s=time.perf_counter() - t_inicio,
        tiempo_entrenamiento_ml_s=selector.tiempo_entrenamiento_s,
        tiempo_inferencia_ml_s=selector.tiempo_inferencia_s,
        modelo_ml=selector,
    )

### Benchmark ALNS (selección uniforme, base) vs ALNS+adaptativo

Compara **ALNS** con selección uniforme de operador contra **ALNS+adaptativo**
(pesos por segmentos vía `SelectorOperadorALNS`), sobre `wlp01`–`wlp04`, 5 semillas
por instancia, `n_iter=10000` para ambos (comparten el mismo tipo de iteración
—destruir + reparar—, así que un mismo presupuesto de iteraciones da comparaciones
comparables, igual que en TS).

In [369]:
import os as _os_alns
import time as _talns
import numpy as np

_EJECUTAR_BENCHMARK_ALNS_ADAPT = False  # cambia a True para ejecutar este benchmark (lento)

if _EJECUTAR_BENCHMARK_ALNS_ADAPT:

    _INST_ALNS_BENCH    = ["wlp01.dzn", "wlp02.dzn", "wlp03.dzn", "wlp04.dzn"]
    _REF_ALNS_BENCH      = {"wlp01": 28716, "wlp02": 52952, "wlp03": 64296, "wlp04": 84633}
    _SEMILLAS_ALNS_BENCH = [0, 1, 2, 3, 4]
    _N_ITER_ALNS_BENCH   = 10_000

    _alns_res = {
        nombre: {"ALNS": {"gaps": [], "t": []},
                 "ALNS+Adaptativo": {"gaps": [], "t": []}}
        for nombre in [a.replace(".dzn", "") for a in _INST_ALNS_BENCH]
    }

    for archivo in _INST_ALNS_BENCH:
        nombre = archivo.replace(".dzn", "")
        datos  = leer_instancia_dzn(_os_alns.path.join("Instaces_MS-CFLP-CI/Public", archivo))
        ref    = _REF_ALNS_BENCH[nombre]

        print(f"\n  {nombre}")
        for semilla in _SEMILLAS_ALNS_BENCH:
            # ── ALNS base (selección uniforme de operador) ────────────────────
            t0 = _talns.perf_counter()
            r_base = alns_base(datos, n_iter=_N_ITER_ALNS_BENCH, semilla=semilla)
            t_base   = _talns.perf_counter() - t0
            gap_base = 100 * (r_base.costo_final - ref) / ref
            _alns_res[nombre]["ALNS"]["gaps"].append(gap_base)
            _alns_res[nombre]["ALNS"]["t"].append(t_base)

            # ── ALNS + selección adaptativa de operador ───────────────────────
            t0 = _talns.perf_counter()
            r_ml = alns_ml(datos, n_iter=_N_ITER_ALNS_BENCH, semilla=semilla)
            t_ml   = _talns.perf_counter() - t0
            gap_ml = 100 * (r_ml.costo_final - ref) / ref
            _alns_res[nombre]["ALNS+Adaptativo"]["gaps"].append(gap_ml)
            _alns_res[nombre]["ALNS+Adaptativo"]["t"].append(t_ml)

            print(f"    seed={semilla}  ALNS: {gap_base:+.2f}% {t_base:.1f}s  "
                  f"ALNS+Adapt: {gap_ml:+.2f}% {t_ml:.1f}s  Δgap={gap_ml-gap_base:+.2f}pp")

    # ── Tabla resumen ─────────────────────────────────────────────────────────────
    instancias_alns = [a.replace(".dzn", "") for a in _INST_ALNS_BENCH]
    print(f"\n{'='*74}")
    print(f"  {'':<10}  {'── ALNS base ──':^20}  {'── ALNS+Adapt ──':^20}  {'Δgap':>6}")
    print(f"  {'Inst.':<10}  {'gap%':>7}  {'σ':>5}  {'t(s)':>5}  "
          f"{'gap%':>7}  {'σ':>5}  {'t(s)':>5}  {'(pp)':>6}")
    print(f"  {'-'*74}")

    all_base, all_ml = [], []
    for nombre in instancias_alns:
        bg = np.mean(_alns_res[nombre]["ALNS"]["gaps"])
        bs = np.std(_alns_res[nombre]["ALNS"]["gaps"])
        bt = np.mean(_alns_res[nombre]["ALNS"]["t"])
        mg = np.mean(_alns_res[nombre]["ALNS+Adaptativo"]["gaps"])
        ms = np.std(_alns_res[nombre]["ALNS+Adaptativo"]["gaps"])
        mt = np.mean(_alns_res[nombre]["ALNS+Adaptativo"]["t"])
        all_base.append(bg); all_ml.append(mg)
        print(f"  {nombre:<10}  {bg:>+7.2f}  {bs:>5.2f}  {bt:>5.1f}  "
              f"{mg:>+7.2f}  {ms:>5.2f}  {mt:>5.1f}  {mg-bg:>+6.2f}")

    print(f"  {'-'*74}")
    mb = np.mean(all_base); mm = np.mean(all_ml)
    print(f"  {'Media':<10}  {mb:>+7.2f}  {'':<5}  {'':<5}  {mm:>+7.2f}  {'':<5}  {'':<5}  {mm-mb:>+6.2f}")
    print(f"  {'='*74}")
else:
    print("Benchmark ALNS vs ALNS+Adaptativo omitido (_EJECUTAR_BENCHMARK_ALNS_ADAPT = False)")

Benchmark ALNS vs ALNS+Adaptativo omitido (_EJECUTAR_BENCHMARK_ALNS_ADAPT = False)


## 6. Benchmark sobre instancias MS-CFLP-CI

Ejecuta los ocho algoritmos (4 pares base/ML) sobre las instancias `wlp01`–`wlp08`
de `Instaces_MS-CFLP-CI/Public/`, con **10 semillas** por instancia, y compara los
costes obtenidos contra la referencia de CPLEX/Gurobi (óptimo probado en
`wlp01`–`wlp03`, mejor valor conocido en el resto, marcado con `*`).

**Presupuesto de tiempo compartido entre familias.** Las cuatro familias (y
cada par base/+ML) ya no se comparan con un número de iteraciones fijo por
algoritmo —antes GRASP recibía 5 iteraciones (~0.1s) y SA 500 000 movimientos
(~14s) sin relación entre sí, una comparación injusta entre familias aunque sí
justa dentro de cada par—, sino con el **mismo presupuesto de tiempo de reloj**
(`_PRESUPUESTO_BASE_S`, 15s por defecto para `wlp01`). Ese presupuesto se
escala linealmente con el número de clientes de cada instancia respecto a
`wlp01` (`_C_REFERENCIA_PRESUPUESTO = 115`), ya que instancias más grandes
tienen soluciones más complejas y necesitan más tiempo para explorarlas.
GVNS es la excepción esperable: al converger a un óptimo local respecto a los
tres vecindarios puede parar antes de agotar su presupuesto —es el criterio de
parada estándar de VNS, no un fallo—, mientras que GRASP, SA y ALNS siempre
agotan el tiempo asignado.

Las instancias tienen tamaños crecientes: desde 50 almacenes / 115 tiendas (`wlp01`)
hasta 500 / 1277 (`wlp08`). Con 10 semillas × 8 instancias × 8 algoritmos y un
presupuesto que crece con cada instancia, el benchmark completo es lento;
está controlado por el flag `_EJECUTAR_BENCHMARK_FINAL` de la celda siguiente.
Al final de la sección se muestra, además de las tablas de resultados, un
resumen de lo que aprendió cada componente de ML sobre cada instancia.


In [370]:
_C_REFERENCIA_PRESUPUESTO = 115  # nº de clientes de wlp01, la instancia de referencia

def benchmarcar_instancia(
    datos: DatosCFLP, grado_poly: int = 2, semilla: int = 0, presupuesto_s: float = 15.0,
) -> dict[str, ResultadoBenchmark]:
    """Ejecuta los ocho algoritmos (base y +ML) sobre una instancia y devuelve
    un diccionario {nombre: ResultadoBenchmark} listo para tabular.

    Las cuatro familias (y cada par base/+ML dentro de ellas) comparten el
    mismo presupuesto de tiempo de reloj, en vez de un número de iteraciones
    fijo por algoritmo: antes GRASP recibía 5 iteraciones (~0.1s) y SA 500 000
    movimientos (~14s) sin relación entre sí, así que no era una comparación
    justa entre familias. `presupuesto_s` es el presupuesto para la instancia
    de referencia (wlp01, `_C_REFERENCIA_PRESUPUESTO` clientes) y se escala
    linealmente con el número de clientes de `datos`, ya que instancias más
    grandes tienen soluciones más complejas y necesitan más tiempo para
    explorarlas."""
    resultados: dict[str, ResultadoBenchmark] = {}

    F, C  = datos["numero_instalaciones"], datos["numero_clientes"]
    cap   = datos["capacidad_instalacion"]
    cf    = datos["costo_fijo_instalacion"]
    dem   = datos["demanda_cliente"]
    envio = datos["costo_envio"]
    pares_inc = datos["pares_incompatibles"]

    presupuesto = presupuesto_s * (C / _C_REFERENCIA_PRESUPUESTO)

    # ── GRASP base / GRASP+ML ──────────────────────────────────────────────
    t0 = time.perf_counter()
    with contextlib.redirect_stdout(io.StringIO()):
        r_grasp = ejecutar_grasp(
            F, C, cap, cf, dem, envio,
            alfa=0.5, numero_maximo_de_iteraciones=1_000_000, semilla_aleatoria=semilla,
            usar_alfa_adaptativo=False, pares_incompatibles=pares_inc,
            tiempo_limite_s=presupuesto,
        )
    resultados["GRASP"] = ResultadoBenchmark(
        nombre="GRASP",
        costo_final=r_grasp["costo_total_minimo"],
        tiempo_total_s=time.perf_counter() - t0,
    )

    resultados["GRASP+ML"] = ejecutar_grasp_ml(
        datos, grado=grado_poly, n_iter=1_000_000, min_muestras=15, semilla=semilla,
        verbose=False, tiempo_limite_s=presupuesto,
    )

    # ── GVNS base (VND fijo) / GVNS+QL (Q-learning) ────────────────────────
    incompatibles = _precomputar_incompatibles(C, pares_inc)
    ab0, asig0, rest0 = seleccionar_solucion_aleatoria_del_pool(datos, tamano_pool=5, semilla_aleatoria=semilla)
    ab0, asig0, rest0 = mejorar_solucion_con_busqueda_local(
        ab0, asig0, rest0, F, C, cap, cf, dem, envio, incompatibles,
    )

    t0 = time.perf_counter()
    costo_gvns = _ejecutar_gvns_iter_vnd(
        datos, ab0, asig0, rest0, k_maximo=3, n_iter=100_000, semilla=semilla,
        tiempo_limite_s=presupuesto,
    )
    resultados["GVNS"] = ResultadoBenchmark(
        nombre="GVNS", costo_final=costo_gvns, tiempo_total_s=time.perf_counter() - t0,
    )

    t0 = time.perf_counter()
    costo_gvns_ql, agente_ql = _ejecutar_gvns_iter_ql(
        datos, ab0, asig0, rest0, k_maximo=3, n_iter=100_000, semilla=semilla,
        tiempo_limite_s=presupuesto,
    )
    resultados["GVNS+QL"] = ResultadoBenchmark(
        nombre="GVNS+QL", costo_final=costo_gvns_ql, tiempo_total_s=time.perf_counter() - t0,
        modelo_ml=agente_ql,
    )

    # ── SA base (movimientos) / SA+UCB1 ────────────────────────────────────
    resultados["SA"] = recocido_simulado(
        datos, n_movimientos=100_000_000, semilla=semilla, tiempo_limite_s=presupuesto,
    )
    resultados["SA+ML"] = recocido_simulado_ml(
        datos, n_movimientos=100_000_000, semilla=semilla, tiempo_limite_s=presupuesto,
    )

    # ── ALNS base (selección uniforme) / ALNS+Adaptativo ───────────────────
    resultados["ALNS"] = alns_base(
        datos, n_iter=10_000_000, semilla=semilla, tiempo_limite_s=presupuesto,
    )
    resultados["ALNS+Adaptativo"] = alns_ml(
        datos, n_iter=10_000_000, tamano_segmento=100, semilla=semilla, tiempo_limite_s=presupuesto,
    )

    return resultados

In [371]:
# Valores de referencia por solver (None = no disponible / timeout)
_CPLEX = {
    "wlp01": 28716, "wlp02": 52952, "wlp03": 64296,
    "wlp04": None,  "wlp05": None,  "wlp06": None,
    "wlp07": None,  "wlp08": None,
}
_GUROBI = {
    "wlp01": 28716, "wlp02": 52952, "wlp03": 64296,
    "wlp04": 84633, "wlp05": 103857, "wlp06": 111654,
    "wlp07": 162277, "wlp08": 187938,
}

# Referencia = mínimo de los valores disponibles
REFERENCIA = {
    inst: min(v for v in [_CPLEX[inst], _GUROBI[inst]] if v is not None)
    for inst in _CPLEX
}

def _tag(inst):
    c, g = _CPLEX[inst], _GUROBI[inst]
    if c is not None and g is not None and c == g:
        return "✓"   # óptimo probado (ambos solvers coinciden)
    return "*"       # mejor conocido


In [ ]:
_EJECUTAR_BENCHMARK_FINAL = True  # cambia a True para ejecutar el benchmark final (lento: 10 semillas x 8 instancias x 8 algoritmos)

_PRESUPUESTO_BASE_S = 3.0  # presupuesto de tiempo (s) compartido entre las 4 familias, para wlp01; se escala con el nº de clientes en las demás instancias
_SEMILLAS_BENCHMARK_FINAL = list(range(10))
_ALGORITMOS_BENCHMARK = ["GRASP", "GRASP+ML", "GVNS", "GVNS+QL", "SA", "SA+ML", "ALNS", "ALNS+Adaptativo"]
_PARES_BENCHMARK = [("GRASP", "GRASP+ML"), ("GVNS", "GVNS+QL"), ("SA", "SA+ML"), ("ALNS", "ALNS+Adaptativo")]

INSTANCIAS_BENCHMARK = [
    "wlp01.dzn", "wlp02.dzn", "wlp03.dzn", "wlp04.dzn",
    "wlp05.dzn", "wlp06.dzn", "wlp07.dzn", "wlp08.dzn",
]
data_dir = "Instaces_MS-CFLP-CI/Public"

resultados_benchmark: dict[str, dict] = {}
modelos_ml: dict[str, dict[str, object]] = {}

if _EJECUTAR_BENCHMARK_FINAL:
    for archivo in INSTANCIAS_BENCHMARK:
        nombre = archivo.replace(".dzn", "")
        datos  = leer_instancia_dzn(os.path.join(data_dir, archivo))
        F, C   = datos["numero_instalaciones"], datos["numero_clientes"]
        ref    = REFERENCIA[nombre]

        gaps    = {alg: [] for alg in _ALGORITMOS_BENCHMARK}
        tiempos = {alg: [] for alg in _ALGORITMOS_BENCHMARK}

        print(f"\n{nombre} {_tag(nombre)}  (F={F}, C={C})")
        for semilla in _SEMILLAS_BENCHMARK_FINAL:
            res = benchmarcar_instancia(
                datos, grado_poly=_MEJOR_GRADO, semilla=semilla, presupuesto_s=_PRESUPUESTO_BASE_S,
            )
            for alg in _ALGORITMOS_BENCHMARK:
                gaps[alg].append(100 * (res[alg].costo_final - ref) / ref)
                tiempos[alg].append(res[alg].tiempo_total_s)
            if semilla == 0:
                modelos_ml[nombre] = {
                    alg: res[alg].modelo_ml for alg in _ALGORITMOS_BENCHMARK
                    if res[alg].modelo_ml is not None
                }
            print(f"  seed={semilla}  " + "  ".join(
                f"{alg}={gaps[alg][-1]:+.1f}%/{tiempos[alg][-1]:.1f}s" for alg in _ALGORITMOS_BENCHMARK
            ), flush=True)

        resultados_benchmark[nombre] = {"F": F, "C": C, "gaps": gaps, "tiempos": tiempos}
else:
    print("Benchmark final omitido (_EJECUTAR_BENCHMARK_FINAL = False)")



wlp01 ✓  (F=50, C=115)


KeyboardInterrupt: 

In [ ]:
if resultados_benchmark:
    col_w = 9

    # ── Gap (%) medio ± desviación estándar sobre 10 semillas ──
    print("Gap medio respecto a la referencia (%, media ± σ sobre 10 semillas):")
    header = f"{'Instancia':<10}"
    for alg in _ALGORITMOS_BENCHMARK:
        header += f"  {alg:>13}"
    print(header)
    print("-" * len(header))

    medias_por_alg = {alg: [] for alg in _ALGORITMOS_BENCHMARK}
    for nombre, datos_r in resultados_benchmark.items():
        line = f"{nombre+' '+_tag(nombre):<10}"
        for alg in _ALGORITMOS_BENCHMARK:
            g = datos_r["gaps"][alg]
            media, sigma = np.mean(g), np.std(g)
            medias_por_alg[alg].append(media)
            line += f"  {media:>6.2f}±{sigma:<5.2f}"
        print(line)
    print("-" * len(header))
    line = f"{'Media':<10}"
    for alg in _ALGORITMOS_BENCHMARK:
        line += f"  {np.mean(medias_por_alg[alg]):>13.2f}"
    print(line)

    # ── Mejora ML sobre base (Δgap en pp) ──
    print()
    print("Mejora del algoritmo ML sobre su base (Δgap en pp, media sobre 10 semillas; + = ML mejora):")
    header2 = f"{'Instancia':<10}"
    for base, ml in _PARES_BENCHMARK:
        header2 += f"  {base+' vs '+ml:>18}"
    print(header2)
    print("-" * len(header2))

    deltas_por_par = {(base, ml): [] for base, ml in _PARES_BENCHMARK}
    for nombre, datos_r in resultados_benchmark.items():
        line = f"{nombre+' '+_tag(nombre):<10}"
        for base, ml in _PARES_BENCHMARK:
            delta = np.mean(datos_r["gaps"][base]) - np.mean(datos_r["gaps"][ml])
            deltas_por_par[(base, ml)].append(delta)
            line += f"  {delta:>15.2f} pp"
        print(line)
    print("-" * len(header2))
    line = f"{'Media':<10}"
    for base, ml in _PARES_BENCHMARK:
        line += f"  {np.mean(deltas_por_par[(base, ml)]):>15.2f} pp"
    print(line)
else:
    print("No hay resultados. Activa _EJECUTAR_BENCHMARK_FINAL y ejecuta la celda anterior.")


No hay resultados. Activa _EJECUTAR_BENCHMARK_FINAL y ejecuta la celda anterior.


### Qué aprendieron los modelos de ML

Para cada instancia se guarda el modelo/agente entrenado en la semilla 0 (una
de las 10 usadas en el benchmark), a modo de ejemplo interpretable de lo que
cada componente de ML aprendió sobre esa instancia concreta — no es un
promedio sobre las 10 semillas, solo una muestra representativa de una
ejecución real.


In [ ]:
if modelos_ml:
    for nombre, modelos in modelos_ml.items():
        print(f"\n{'='*70}")
        print(f"  {nombre}")
        print(f"{'='*70}")

        if "GRASP+ML" in modelos:
            selector = modelos["GRASP+ML"]
            alfa_aprendido = selector.seleccionar_alfa()
            print(f"  GRASP+ML  (regresión polinómica grado {selector.grado}, "
                  f"{len(selector.historial_alfa)} muestras acumuladas)")
            print(f"    alfa* recomendado por el modelo final: {alfa_aprendido:.3f}")

        if "GVNS+QL" in modelos:
            print(f"  GVNS+QL")
            print(f"    {modelos['GVNS+QL'].resumen()}")

        if "SA+ML" in modelos:
            selector = modelos["SA+ML"]
            print(f"  SA+UCB1  (estadísticas finales por brazo)")
            for ratio, veces, suma in zip(selector.RATIOS, selector.conteo, selector.suma_recompensa):
                media = suma / veces if veces > 0 else float("nan")
                print(f"    ratio={ratio:<6} veces_elegido={veces:<4} recompensa_media={media:.4f}")

        if "ALNS+Adaptativo" in modelos:
            selector = modelos["ALNS+Adaptativo"]
            print(f"  ALNS+Adaptativo  (pesos finales)")
            nombres_destroy = ["D1 Aleatorio", "D2 Peor coste", "D3 Relacionado", "D4 Cierre instalación"]
            nombres_repair  = ["R1 Greedy", "R2 Regret-2"]
            for n, w in zip(nombres_destroy, selector.pesos_destroy):
                print(f"    {n:<24} peso={w:.2f}")
            for n, w in zip(nombres_repair, selector.pesos_repair):
                print(f"    {n:<24} peso={w:.2f}")
else:
    print("No hay modelos guardados. Ejecuta primero el benchmark (_EJECUTAR_BENCHMARK_FINAL = True).")


No hay modelos guardados. Ejecuta primero el benchmark (_EJECUTAR_BENCHMARK_FINAL = True).
